In [ ]:
import pandas as pd
'''
Cardiac_G2P_cleaned_HCM_syndromic.csv is Cardiac_G2P_cleaned.csv (the
gene-symbol/referral-indication-renamed export of the raw Cardiac_G2P.csv)
plus one added column, "Syndromic or non-syndromic HCM"

Loading it directly as the single source avoids
keeping two files with duplicate rows in sync with each other.
'''

g2p_clean = pd.read_csv("Cardiac_G2P_cleaned_HCM_syndromic.csv", encoding="utf-8")
g2p_clean.columns = g2p_clean.columns.str.strip()
g2p_clean = g2p_clean.rename(columns={"Syndromic or non-syndromic HCM": "hcm_subtype"})

for col in g2p_clean.columns:
    if g2p_clean[col].dtype == "object":
        g2p_clean[col] = g2p_clean[col].astype(str).str.strip()

# "NA" (string, from the source CSV) means "not an HCM row", not missing data
g2p_clean["hcm_subtype"] = g2p_clean["hcm_subtype"].replace({"NA": None, "nan": None, "": None})

# Standardize gene validity text
g2p_clean["gene_disease_validity"] = (
    g2p_clean["gene_disease_validity"]
    .str.lower()
    .replace({
        "definitive": "Definitive",
        "strong": "Strong",
        "moderate": "Moderate",
        "limited": "Limited",
        "disputed": "Disputed",
        "refuted": "Refuted",
        "no known disease relationship": "No known disease relationship",
    })
)

if "entry_date" in g2p_clean.columns:
    g2p_clean["entry_date"] = pd.to_datetime(g2p_clean["entry_date"], errors="coerce")

print("Loaded:", "Cardiac_G2P_cleaned_HCM_syndromic.csv")
print("Rows:", len(g2p_clean))
print("Columns:", g2p_clean.columns.tolist())
print("\nGene validity counts:")
print(g2p_clean["gene_disease_validity"].value_counts(dropna=False))
print("\nHCM subtype counts:")
print(g2p_clean["hcm_subtype"].value_counts(dropna=False))


In [ ]:
# --------------------------------------------------
# Disease reference (built from Cardiac_G2P's own referral indications)
# --------------------------------------------------
'''
Previously sourced from disease_reference.csv (the legacy Perl tool's
20-disease list from the legacy tool's internal database) - most of which
Cardiac_G2P has no gene-disease pairs for at all. Replaced with the 7
referral indications Cardiac_G2P actually curates, read directly off
g2p_clean so the disease menu shown in mainn.ipynb can never drift out of
sync with what validate_gene_disease_pair can actually check.

Hypertrophic Cardiomyopathy is split into two menu entries -- familial
(non-syndromic) and syndromic -- using the hcm_subtype column added above,
since the two are mechanistically distinct referral reasons even though
they share one Cardiac_G2P referral_indication string.
'''
import re


def _short_code(referral):
    m = re.search(r"\(([^)]+)\)\s*$", referral)
    return m.group(1) if m else referral


_referrals = sorted(g2p_clean["referral_indication"].dropna().unique())

_disease_rows = []
for _referral in _referrals:
    _code = _short_code(_referral)
    _long_name = re.sub(r"\s*\([^)]+\)\s*$", "", _referral).strip()
    if _code == "HCM":
        _disease_rows.append({
            "dis_name": "HCM-FAM",
            "dis_long_name": f"{_long_name} - familial (non-syndromic)",
            "referral_indication": _referral,
            "hcm_subtype": "Non-syndromic",
        })
        _disease_rows.append({
            "dis_name": "HCM-SYN",
            "dis_long_name": f"{_long_name} - syndromic",
            "referral_indication": _referral,
            "hcm_subtype": "Syndromic",
        })
    else:
        _disease_rows.append({
            "dis_name": _code,
            "dis_long_name": _long_name,
            "referral_indication": _referral,
            "hcm_subtype": None,
        })

disease_ref = pd.DataFrame(_disease_rows)


def validate_gene_disease_pair(gene_symbol, disease_short_code, g2p_df, disease_ref_df=disease_ref):
    """
    Check whether (gene_symbol, disease_short_code) is a real, curated
    gene-disease pair in Cardiac_G2P. disease_short_code is one of
    disease_ref's dis_name codes (e.g. "HCM-FAM", "HCM-SYN", "DCM", "LQTS").

    For HCM-FAM/HCM-SYN, the match additionally requires the gene's
    hcm_subtype in Cardiac_G2P to agree with the selected subtype -- a gene
    curated only as Syndromic HCM should not validate against a "familial"
    selection, and vice versa.
    """

    result = {
        "disease_short_code":      disease_short_code,
        "g2p_referral_indication": None,
        "pair_found":              False,
        "reason":                  None,
    }

    if not gene_symbol or not disease_short_code:
        result["reason"] = "gene or disease not provided"
        return result

    ref_row = disease_ref_df[disease_ref_df["dis_name"] == disease_short_code]
    if ref_row.empty:
        result["reason"] = (
            f"{disease_short_code} is not covered by Cardiac_G2P "
            f"(no gene-disease pairs curated for it)"
        )
        return result
    referral    = ref_row.iloc[0]["referral_indication"]
    hcm_subtype = ref_row.iloc[0]["hcm_subtype"]
    result["g2p_referral_indication"] = referral

    gene_rows = g2p_df[g2p_df["gene_symbol"].astype(str).str.upper() == str(gene_symbol).upper()]
    if gene_rows.empty:
        result["reason"] = f"{gene_symbol} is not in Cardiac_G2P at all"
        return result

    match = gene_rows[gene_rows["referral_indication"] == referral]
    if hcm_subtype is not None:
        match = match[match["hcm_subtype"] == hcm_subtype]

    if match.empty:
        if hcm_subtype is not None and not gene_rows[gene_rows["referral_indication"] == referral].empty:
            gene_hcm_rows = gene_rows[gene_rows["referral_indication"] == referral]
            other_subtypes = ", ".join(sorted(gene_hcm_rows["hcm_subtype"].dropna().unique())) or "none curated"
            result["reason"] = (
                f"{gene_symbol} is curated for {referral}, but not as {hcm_subtype} HCM "
                f"-- it's curated as: {other_subtypes}"
            )
        else:
            other = ", ".join(sorted(gene_rows["referral_indication"].dropna().unique()))
            result["reason"] = (
                f"{gene_symbol} is in Cardiac_G2P, but not for {disease_short_code} "
                f"-- it's associated with: {other}"
            )
        return result

    result["pair_found"] = True
    return result


def lookup_g2p_by_gene_disease(gene_symbol, disease_short_code, g2p_df, disease_ref_df=disease_ref):
    """
    Scope Cardiac_G2P hits to the confirmed disease, not just the gene.

    A gene's mechanism/allelic-requirement/penetrance facts in Cardiac_G2P
    are disease-specific: FLNC is loss-of-function for DCM but not
    established LOF for HCM; KCNQ1 is loss-of-function for LQTS but
    gain-of-function for Short QT Syndrome. Every rule that reads
    context["g2p_hits"] (PVS1's LOF check, BP1, BS2/BP2 penetrance gating,
    PM3/BP2's biallelic check, BA1/BS1's zygosity adjustment) must only see
    the row(s) for the disease actually being classified - not every
    disease this gene happens to be curated for elsewhere in G2P. Returns an
    empty frame if the disease isn't recognised or the gene has no rows for it.

    """
    all_hits = lookup_g2p_by_gene(gene_symbol, g2p_df)
    if all_hits.empty or not disease_short_code:
        return all_hits.iloc[0:0].copy()

    ref_row = disease_ref_df[disease_ref_df["dis_name"] == disease_short_code]
    if ref_row.empty:
        return all_hits.iloc[0:0].copy()

    referral    = ref_row.iloc[0]["referral_indication"]
    hcm_subtype = ref_row.iloc[0]["hcm_subtype"]

    scoped = all_hits[all_hits["referral_indication"] == referral]
    if hcm_subtype is not None:
        scoped = scoped[scoped["hcm_subtype"] == hcm_subtype]
    return scoped


In [ ]:
# --------------------------------------------------
# Hotspot / domain reference data (PM1, PP2)
# --------------------------------------------------

'''
# Equivalent of the old ICC domain_analysis/domain_data tables, but with the
# Fisher's-p / enrichment-factor threshold check already resolved into a
# rule ("PM1"/"PP2"/NULL) + strength ("moderate"/"supporting") call.
'''

hotspot = pd.read_csv("hotspot_regions.csv", encoding="utf-8")
hotspot.columns = hotspot.columns.str.strip()

hotspot["gene"] = hotspot["gene"].astype(str).str.strip().str.upper()
hotspot["disease"] = hotspot["disease"].astype(str).str.strip()
hotspot["rule"] = hotspot["rule"].astype(str).str.strip()
hotspot["strength"] = hotspot["strength"].astype(str).str.strip().str.lower()

for col in ("aa_start", "aa_end"):
    hotspot[col] = pd.to_numeric(hotspot[col], errors="coerce")

hotspot_clean = hotspot.copy()

print("Loaded:", "hotspot_regions.csv")
print("Rows:", len(hotspot_clean))
print("Genes covered:", sorted(hotspot_clean["gene"].unique().tolist()))


def lookup_hotspot_by_gene(gene_symbol, hotspot_df):
    if not gene_symbol:
        return hotspot_df.iloc[0:0].copy()
    return hotspot_df[
        hotspot_df["gene"].astype(str).str.upper() == str(gene_symbol).upper()
    ].copy()


In [ ]:
import json
from pathlib import Path

def load_annotation(json_path):
    with Path(json_path).open("r", encoding="utf-8") as f:
        return json.load(f)


def get_canonical_transcript(annotation):
    """
    Prefer a canonical transcript that VEP actually computed a real coding
    consequence for (non-null hgvsc) over one that is merely flagged
    canonical but structurally uninvolved (e.g. upstream_gene_variant on an
    overlapping/adjacent gene). Ensembl can mark canonical=1 on transcripts
    from TWO different genes at the same locus when genes overlap (found via
    NM_004415.4(DSP):c.8526_8537del, whose canonical list included both
    DSP/ENST00000379802 - the real inframe_deletion, hgvsc populated - and
    SNRNP48/ENST00000342415 - upstream_gene_variant, hgvsc=None - with
    SNRNP48 appearing first in VEP's own transcript order). The old
    canonical[0] pick silently took the wrong gene whenever this happened.
    """
    transcripts = annotation.get("transcripts", [])
    if not transcripts:
        return None
    canonical = [tx for tx in transcripts if tx.get("canonical") == 1]
    if not canonical:
        return transcripts[0]
    with_hgvsc = [tx for tx in canonical if tx.get("hgvsc")]
    return with_hgvsc[0] if with_hgvsc else canonical[0]


def lookup_g2p_by_gene(gene_symbol, g2p_df):
    if not gene_symbol:
        return g2p_df.iloc[0:0].copy()
    return g2p_df[
        g2p_df["gene_symbol"].astype(str).str.upper() == str(gene_symbol).upper()
    ].copy()


def build_classification_context(annotation, g2p_df):
    tx          = get_canonical_transcript(annotation)
    gene_symbol = tx.get("gene_symbol") if tx else None
    hits        = lookup_g2p_by_gene(gene_symbol, g2p_df)
    pop         = annotation.get("population_summary") or {}

    return {
        "gene_symbol":        gene_symbol,
        "chosen_transcript":  tx,
        "annotation":         annotation,
        "g2p_hits":           hits,
        "icc_gene":           not hits.empty,
        "population_summary": pop,
        # gnomAD FAF95 — surfaced explicitly for ACMG rules
        "gnomad_variant_id":  pop.get("gnomad_variant_id"),
        "faf95_popmax":       pop.get("faf95_popmax"),
        "faf95_population":   pop.get("faf95_population"),
        "faf_source":         pop.get("faf_source"),
        # in silico scores
        "revel":              annotation.get("revel"),
        "spliceai":           annotation.get("spliceai"),
    }

In [ ]:
def build_classification_context(annotation, g2p_df, disease_short_code=None):
    tx           = get_canonical_transcript(annotation)
    gene_symbol  = tx.get("gene_symbol") if tx else None
    all_g2p_hits = lookup_g2p_by_gene(gene_symbol, g2p_df)
    gene_disease_pair = validate_gene_disease_pair(gene_symbol, disease_short_code, g2p_df)
    # g2p_hits (consumed by every mechanism/penetrance rule) is scoped to the
    # CONFIRMED disease only -- see lookup_g2p_by_gene_disease. If the pair
    # isn't confirmed, we don't know which of this gene's G2P entries (if
    # any) actually applies, so no mechanism data is handed to the rules
    # rather than guessing from an unrelated disease's entry.
    g2p_hits = (
        lookup_g2p_by_gene_disease(gene_symbol, disease_short_code, g2p_df)
        if gene_disease_pair["pair_found"] else all_g2p_hits.iloc[0:0].copy()
    )
    pop          = annotation.get("population_summary") or {}

    return {
        "gene_symbol":        gene_symbol,
        "chosen_transcript":  tx,
        "annotation":         annotation,
        "g2p_hits":           g2p_hits,
        "all_g2p_hits":       all_g2p_hits,
        "icc_gene":           not all_g2p_hits.empty,
        "disease_short_code": disease_short_code,
        "gene_disease_pair":  gene_disease_pair,
        "population_summary": pop,
        "gnomad_variant_id":  pop.get("gnomad_variant_id"),
        "faf95_popmax":       pop.get("faf95_popmax"),
        "faf95_population":   pop.get("faf95_population"),
        "faf_source":         pop.get("faf_source"),
        "revel_score":        annotation.get("revel_score"),
        "revel":              annotation.get("revel"),
        "spliceai":           annotation.get("spliceai"),
        "spliceai_score":     annotation.get("spliceai_score"),
    }


# ACMG Rule application

ClinGen CSpec Registry: https://cspec.genome.network/cspec/ui/svi/


Cardiomyopathy VCEP Specifications: https://www.clinicalgenome.org/docs/clingen-cardiomyopathy-expert-panel-specifications-to-the-acmg-amp-variant-interpretation-guidelines-version-1/

# Population Frequency logic


In [ ]:
import re
import requests

CSPEC_API = "https://cspec.genome.network/cspec/api"

GENERIC_ACMG_THRESHOLDS = {
    "ba1":     0.05,
    "bs1":     0.01,
    "pm2_max": 1e-4,
    "pm2_strength": "supporting",  # ClinGen SVI general recommendation (downgraded
                                    # from the original 2015 Moderate), and matches this
                                    # VCEP's own choice for every gene checked in the CSpec registry
    "pp3_min": 0.75,
    "bp4_max": 0.15,
    "bp7_max": 0.10,
    "applicability": {},
    "source":  "generic ACMG (Richards 2015)",
}

_gene_cspec_map  = None
_threshold_cache = {}


def _load_gene_cspec_map():
    try:
        resp = requests.get(f"{CSPEC_API}/svis", timeout=30)
        resp.raise_for_status()
        entries = resp.json().get("data", [])   # /svis returns {"data": [...]}
    except Exception as exc:
        print(f"CSpec registry fetch warning: {exc}")
        return {}

    mapping = {}
    for entry in entries:
        if not isinstance(entry, dict):
            continue
        if entry.get("status", "").lower() != "released":
            continue
        spec_id = entry.get("@id", "").rstrip("/").split("/")[-1]
        for rs in entry.get("ruleSets", []):
            for gene in rs.get("genes", []):
                label = (gene.get("label") or "").upper().strip()
                if label and spec_id:
                    mapping[label] = spec_id
    return mapping


def _extract_threshold(text, operator):
    """
    Extract the numeric threshold CSpec actually intends, not just the first
    <op>NUMBER substring in the description. Some CSpec text (e.g. MYH7 PM2)
    contains an earlier, UNRELATED number using the same comparison operator
    in surrounding prose (case-frequency justification text) before the real,
    bolded threshold sentence ("A threshold of **<=0.00004**... activates
    this rule"). A naive first-match regex silently returns the wrong,
    500x-too-permissive value in that case.

    Fix: prefer a match anchored to the phrase "threshold of ... <op>NUMBER"
    (the consistent phrasing CSpec uses for the actual activating cutoff),
    and only fall back to a bare first-match if that anchor isn't found.
    """
    op_pattern = re.escape(operator)
    anchored = re.search(
        r"threshold of[^.\n]{0,25}" + op_pattern + r"\s*([0-9]+(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?)",
        text, re.IGNORECASE,
    )
    if anchored:
        return float(anchored.group(1))
    m = re.search(op_pattern + r"\s*([0-9]+(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?)", text)
    return float(m.group(1)) if m else None


def _extract_applicability(rs_data):
    """
    Map rule label -> True if ClinGen CSpec marks ANY evidence-strength entry
    for that rule as "Applicable" for this gene's VCEP ruleset, False if every
    entry is explicitly "Not applicable".

    This matters beyond just a threshold lookup: some rules are marked
    Not Applicable outright for an entire gene regardless of the variant, e.g.
    for all 8 ClinGen Cardiomyopathy VCEP genes, BP1 and BP3 are "Not
    Applicable" (their HCM mechanism is dominant-negative missense, not
    haploinsufficiency), and PP2 is "Applicable" only for TPM1. Callers should
    treat True/False as an authoritative gene-specific override, and a missing
    key (rule not present in this VCEP's ruleset, or no VCEP spec for this
    gene at all) as "no CSpec constraint -- use generic ACMG logic".
    """
    applicability = {}
    for code in rs_data.get("criteriaCodes", []):
        label = code.get("label", "").upper()
        strengths = code.get("evidenceStrengths", [])
        # Some VCEPs phrase this as "Applicable with VCEP specification" rather
        # than the bare "Applicable" MYH7/MYBPC3 use (e.g. GAA/GN010, Pompe
        # disease) -- match on substring containment (excluding "not
        # applicable") rather than an exact string match, so those aren't
        # silently misread as Not Applicable.
        fired = False
        for s in strengths:
            a = (s.get("applicability") or "").strip().lower()
            if "applicable" in a and "not applicable" not in a:
                fired = True
        applicability[label] = fired
    return applicability


def fetch_cspec_thresholds(gene_symbol):
    global _gene_cspec_map
    gene = (gene_symbol or "").upper().strip()

    if gene in _threshold_cache:
        return _threshold_cache[gene]

    if _gene_cspec_map is None:
        _gene_cspec_map = _load_gene_cspec_map()

    spec_id = _gene_cspec_map.get(gene)
    if not spec_id:
        result = GENERIC_ACMG_THRESHOLDS.copy()
        result["applicability"] = {}
        _threshold_cache[gene] = result
        return result

    thresholds = {"applicability": {}, "pm2_strength": "supporting"}
    try:
        spec_resp = requests.get(
            f"{CSPEC_API}/SequenceVariantInterpretation/id/{spec_id}", timeout=30)
        spec_resp.raise_for_status()
        spec = spec_resp.json()

        for rs_stub in spec.get("ruleSets", []):
            rs_id = rs_stub.get("@id", "").rstrip("/").split("/")[-1]
            rs_resp = requests.get(f"{CSPEC_API}/RuleSet/id/{rs_id}", timeout=30)
            rs_resp.raise_for_status()
            rs_data = rs_resp.json()

            thresholds["applicability"].update(_extract_applicability(rs_data))

            for code in rs_data.get("criteriaCodes", []):
                label = code.get("label", "").upper()
                for strength in code.get("evidenceStrengths", []):
                    if "not applicable" in strength.get("applicability", "").lower():
                        continue
                    desc = strength.get("description", "")
                    if label == "BA1" and "ba1" not in thresholds:
                        val = _extract_threshold(desc, ">=")
                        if val is None:
                            val = _extract_threshold(desc, "≥")
                        if val is not None:
                            thresholds["ba1"] = val
                    elif label == "BS1" and "bs1" not in thresholds:
                        val = _extract_threshold(desc, ">=")
                        if val is None:
                            val = _extract_threshold(desc, "≥")
                        if val is not None:
                            thresholds["bs1"] = val
                    elif label == "PM2" and "pm2_max" not in thresholds:
                        val = _extract_threshold(desc, "<=")
                        if val is None:
                            val = _extract_threshold(desc, "≤")
                        if val is not None:
                            thresholds["pm2_max"] = val
                            thresholds["pm2_strength"] = strength.get("label", "").strip().lower()
                    elif label == "PP3" and "pp3_min" not in thresholds:
                        val = _extract_threshold(desc, ">=")
                        if val is None:
                            val = _extract_threshold(desc, "≥")
                        if val is not None:
                            thresholds["pp3_min"] = val
                    elif label == "BP4" and "bp4_max" not in thresholds:
                        val = _extract_threshold(desc, "<=")
                        if val is None:
                            val = _extract_threshold(desc, "≤")
                        if val is not None:
                            thresholds["bp4_max"] = val
                    elif label == "BP7" and "bp7_max" not in thresholds:
                        # CSpec text for BP7 is qualitative for the Cardiomyopathy
                        # VCEP genes (no numeric cutoff) -- this only takes effect
                        # if some other gene's ruleset ever specifies one.
                        val = _extract_threshold(desc, "<=")
                        if val is None:
                            val = _extract_threshold(desc, "≤")
                        if val is not None:
                            thresholds["bp7_max"] = val

        thresholds["source"] = f"ClinGen CSpec {spec_id}"

    except Exception as exc:
        print(f"CSpec API warning for {gene} ({spec_id}): {exc}")

    result = {**GENERIC_ACMG_THRESHOLDS, **thresholds}
    _threshold_cache[gene] = result
    return result


In [ ]:
_PP3_BP4_ELIGIBLE_CONSEQUENCES = {"missense_variant", "splice_region_variant"}


def _is_revel_eligible(context):
    # PP3/BP4 only apply to missense or splice-region variants
    tx = context.get("chosen_transcript") or {}
    consequences = set(tx.get("consequence_terms", []) or [])
    return bool(consequences & _PP3_BP4_ELIGIBLE_CONSEQUENCES)


_GNOMAD_GLOBAL_KEYS = {"gnomadg", "gnomade", "gnomad", "af"}


def _extract_global_gnomad_af(context):
    """
    Pull the overall (all-population) gnomAD AF from the raw gnomad_frequencies
    stored in the annotation.  Returns (af, source_key) or (None, None).

    gnomad_frequencies structure (from VEP colocated_variants):
      [{"id": "rs...", "allele_string": "G/A",
        "frequencies": {"T": {"gnomadg": 2.6e-5, "gnomadg_sas": 2e-4, ...}}}]

    We only read the GLOBAL keys (gnomadg / gnomade / gnomad / af) to avoid
    inflating the AF from a single high-frequency subpopulation.
    """
    annotation = context.get("annotation") or {}
    gnomad_freqs = annotation.get("gnomad_frequencies") or []

    best_af  = None
    best_src = None

    for entry in gnomad_freqs:
        if not isinstance(entry, dict):
            continue
        for _allele, allele_freqs in (entry.get("frequencies") or {}).items():
            if not isinstance(allele_freqs, dict):
                continue
            for key, value in allele_freqs.items():
                if key.lower() not in _GNOMAD_GLOBAL_KEYS:
                    continue
                try:
                    value = float(value)
                except (TypeError, ValueError):
                    continue
                if value > 0 and (best_af is None or value > best_af):
                    best_af  = value
                    best_src = key

    return best_af, best_src


def _g2p_is_biallelic(g2p_hits):
    """
    True if Cardiac_G2P lists this gene's allelic requirement as
    'biallelic_autosomal' (recessive) for at least one disease entry.
    """
    if g2p_hits is None or g2p_hits.empty or "allelic_requirement" not in g2p_hits.columns:
        return False
    reqs = {str(r).strip().lower() for r in g2p_hits["allelic_requirement"]}
    return "biallelic_autosomal" in reqs


def _g2p_penetrance_note(g2p_hits):
    """
    Non-None caveat string if Cardiac_G2P's inheritance_modifiers flags this
    gene/disease as incompletely or age-dependently penetrant, else None.

    Gates BS2 ("observed in a healthy adult ... full penetrance expected at
    an early age") and BP2's dominant-trans clause ("... for a fully
    penetrant dominant gene disorder") -- both explicitly require full
    penetrance per ACMG/AMP 2015, which cardiomyopathy genes routinely do
    NOT have (incomplete, age-related penetrance is the norm, not the
    exception, for HCM/DCM). Rather than asking the user to judge
    penetrance themselves, this reads the fact Cardiac_G2P already curates.
    """
    if g2p_hits is None or g2p_hits.empty or "inheritance_modifiers" not in g2p_hits.columns:
        return None
    notes = {
        str(m).strip() for m in g2p_hits["inheritance_modifiers"]
        if str(m).strip() and str(m).strip().lower() != "nan"
    }
    flagged = [m for m in notes if "incomplete penetrance" in m.lower() or "age-related onset" in m.lower()]
    return "; ".join(sorted(flagged)) if flagged else None


def apply_population_rules(context):
    pop         = context.get("population_summary") or {}
    gene_symbol = context.get("gene_symbol")

    def to_float(v):
        try:
            return None if v in (None, "", "nan") else float(v)
        except (TypeError, ValueError):
            return None

    # Priority 1: FAF95 (Filtering Allele Frequency — ClinGen preferred)
    faf = to_float(pop.get("faf95_popmax"))
    if faf is not None:
        af, af_source = faf, "faf95_gnomad"
    else:
        # Priority 2: Global gnomAD AF (gnomadg / gnomade) from raw freq data
        global_af, global_src = _extract_global_gnomad_af(context)
        if global_af is not None:
            af, af_source = global_af, f"gnomad_global/{global_src}"
        else:
            # Priority 3: max_population_af fallback (may be subpopulation-specific)
            af        = to_float(pop.get("max_population_af"))
            af_source = "max_af_vep"

    thresholds    = fetch_cspec_thresholds(gene_symbol)
    ba1_threshold = thresholds["ba1"]
    bs1_threshold = thresholds["bs1"]
    pm2_max       = thresholds["pm2_max"]
    pm2_strength  = thresholds.get("pm2_strength", "supporting")

    # Zygosity adjustment for recessive (biallelic) genes only. Mirrors Perl's
    # sqrt(threshold) rule for homozygous variants -- Hardy-Weinberg: homozygote
    # frequency = allele_freq^2, so the "too common to be disease-causing" AF
    # cutoff should be relaxed (raised) for a homozygous observation under a
    # recessive model. Perl applied this sqrt adjustment purely based on
    # zygosity, regardless of the gene's actual inheritance mode. That's too
    # broad: CSpec's own MYH7 BA1 text states the fixed 0.001 threshold is
    # "applicable when assessing variants in the context of autosomal dominant
    # cardiomyopathy" -- i.e. NOT meant to be zygosity-adjusted at all for a
    # dominant gene. So here the adjustment is gated on Cardiac_G2P's
    # allelic_requirement being biallelic_autosomal (recessive), which is the
    # biologically correct scope and matters for the ~21 recessive gene/disease
    # pairs in Cardiac_G2P.csv that fall outside the (all-dominant) VCEP panel.
    zygosity   = (context.get("manual_evidence") or {}).get("zygosity")
    biallelic  = _g2p_is_biallelic(context.get("g2p_hits"))
    zyg_adjusted = False
    if biallelic and zygosity == "hom":
        ba1_threshold = ba1_threshold ** 0.5
        bs1_threshold = bs1_threshold ** 0.5
        zyg_adjusted  = True

    rules = {
        "af_used":           af,
        "af_source":         af_source,
        "ba1_threshold":     ba1_threshold,
        "bs1_threshold":     bs1_threshold,
        "pm2_max":           pm2_max,
        "pm2_strength":      pm2_strength,
        "cspec_source":      thresholds.get("source"),
        "zygosity":          zygosity,
        "is_biallelic_gene": biallelic,
        "zygosity_adjusted": zyg_adjusted,
        "BA1": False,
        "BS1": False,
        "PM2": False,
    }

    if af is None or af == 0:
        rules["PM2"] = True
    elif af >= ba1_threshold:
        rules["BA1"] = True
    elif af >= bs1_threshold:
        rules["BS1"] = True
    elif af <= pm2_max:
        rules["PM2"] = True

    return rules


def _extract_revel_score(context):
    # Primary source: VEP's own REVEL plugin (?REVEL=1 in annotation.ipynb)
    score = context.get("revel_score")
    if score is not None:
        try:
            return float(score)
        except (TypeError, ValueError):
            pass
    # Do NOT fall back to context["revel"] — that comes from find_keys_containing()
    # on the raw VEP result, which can accidentally pick up unrelated fields.
    return None


def apply_computational_rules(context):
    gene_symbol = context.get("gene_symbol")
    thresholds  = fetch_cspec_thresholds(gene_symbol)
    pp3_min     = thresholds.get("pp3_min")
    bp4_max     = thresholds.get("bp4_max")
    revel_score = _extract_revel_score(context)
    eligible    = _is_revel_eligible(context)

    rules = {
        "revel_score":           revel_score,
        "pp3_threshold":         pp3_min,
        "bp4_threshold":         bp4_max,
        "cspec_source":          thresholds.get("source"),
        "variant_type_eligible": eligible,
        "PP3": False,
        "BP4": False,
    }

    if not eligible or revel_score is None:
        return rules

    if pp3_min is not None and revel_score >= pp3_min:
        rules["PP3"] = True
    elif bp4_max is not None and revel_score <= bp4_max:
        rules["BP4"] = True

    return rules


--------------------------------------------------
# TTN-specific PVS1 gate
--------------------------------------------------
TTN has no released ClinGen Cardiomyopathy VCEP CSpec (checked live against
the registry this pipeline already queries for other genes -- zero TTN
entries; TTN isn't on the VCEP's Phase 1/2 gene list either). Generic PVS1
is therefore too permissive for it: TTN is enormous and heavily
alternatively-spliced, so a truncating variant is only reliably pathogenic
if the exon it hits is actually included in the heart-expressed transcript.

Data source: `titin_exon.csv` / `titin_transcript.csv` / `titin_exon_transcript.csv`,
extracted verbatim from the legacy Perl tool's own internal database
(titin_exon/titin_transcript/titin_exon_transcript tables), not from
external literature. The Perl source itself isn't in
this repo, only its database, so which PSI column and threshold it
actually used for classification is NOT verified -- this reconstructs a
defensible gate from the schema plus the field-standard PSI>=90% cutoff
(Roberts et al. 2015; see also ClinGen SVI's general PVS1 recommendations,
Abou Tayoun et al. 2018, which covers TTN as a worked example of
location-dependent PVS1 strength). titin_transcript row 1 ("META") is
ENST00000589042 / NM_001267550.1 -- the exact transcript VEP already
resolves as canonical for TTN here so no coordinate
translation is needed: titin_exon's te_meta_start/te_meta_end are already
CDS-relative positions on this pipeline's own chosen transcript.

Two gates implemented, both directly evidenced against real ClinVar-
conflicting TTN variants during development:
1. PSI gate: te_psi_dcm >= 90 (DCM-cohort-specific PSI column -- most
directly relevant to what PVS1 is trying to establish for this
disease). Below this, the exon is skippable and a truncating variant
there isn't reliably pathogenic (e.g. NM_001267550.2(TTN):c.15776-2A>G,
exon PSI=6 -- ClinVar submitters only went as far as "Likely
pathogenic"/"Uncertain significance", never "Pathogenic").
2. Terminal-exon gate: a variant in the transcript's last exon escapes
nonsense-mediated decay (produces a stable truncated protein rather
than triggering degradation) -- a general ACMG PVS1 rule, not
TTN-specific, just directly checkable here from titin_exon's own
ordering (e.g. NM_001267550.2(TTN):c.107777del hits exon 364, the

literal last exon).
NOT implemented (documented limitation, not silently ignored): symmetric-
exon in-frame-skip nuance. A splice donor/acceptor variant at a
"symmetric" exon (te_symmetry == 'S', length a multiple of 3) can result
in clean in-frame exon skipping rather than a frameshift, even at PSI=100
-- e.g. NM_001267550.2(TTN):c.63793+1G>T (exon 307, PSI=100, symmetric):
submitters called this only "Likely pathogenic", consistent with weaker-
than-PVS1 evidence, but properly handling it needs an assessment of
whether the skipped domain is functionally critical, which this pipeline
doesn't have data to make. TTN null variants at high-PSI symmetric exons
will currently still fire PVS1 -- a known, undercounted false-positive
source distinct from the two gates above.

In [ ]:

_TTN_PSI_DCM_THRESHOLD = 90


def _load_titin_exons(path="titin_exon.csv"):
    df = pd.read_csv(path)
    df = df[df["te_meta_start"].notna()].copy()
    df["te_meta_start"] = df["te_meta_start"].astype(int)
    df["te_meta_end"] = df["te_meta_end"].astype(int)
    return df.sort_values("te_meta_start").to_dict("records")


_TITIN_EXONS = _load_titin_exons()
_TITIN_LAST_EXON_ID = max(e["te_id"] for e in _TITIN_EXONS)


def _parse_ttn_cds_pos(hgvsc):
    """
    Returns (pos, side) from an HGVSc string's c.<pos> component.
    side is None for a plain coding position, "acceptor" for a negative
    intronic offset (c.N-k..., splice acceptor -- affects the exon that
    STARTS at N), "donor" for a positive intronic offset (c.N+k..., splice
    donor -- affects the exon that ENDS at N).
    """
    if not hgvsc:
        return None, None
    m = re.search(r"c\.(\d+)([+-]\d+)?", hgvsc)
    if not m:
        return None, None
    pos = int(m.group(1))
    offset = m.group(2)
    if offset is None:
        return pos, None
    return pos, ("donor" if offset.startswith("+") else "acceptor")


def _find_titin_exon(pos, side):
    if pos is None:
        return None
    if side == "acceptor":
        matches = [e for e in _TITIN_EXONS if e["te_meta_start"] == pos]
    elif side == "donor":
        matches = [e for e in _TITIN_EXONS if e["te_meta_end"] == pos]
    else:
        matches = [e for e in _TITIN_EXONS if e["te_meta_start"] <= pos <= e["te_meta_end"]]
    return matches[0] if matches else None


def check_ttn_pvs1_gate(tx):
    """
    Returns a detail dict; "gate_pass" is True only if this TTN null variant
    hits a non-terminal, high-PSI exon. False (gate_pass) does NOT mean the
    variant is benign -- it means PVS1's very-strong LOF evidence isn't
    supported at this exon, per the two checks documented above.
    """
    base = {
        "exon_te_id": None, "exon_domain": None, "psi_dcm": None,
        "is_terminal_exon": None, "gate_pass": False, "reason": None,
    }

    hgvsc = tx.get("hgvsc")
    pos, side = _parse_ttn_cds_pos(hgvsc)
    if pos is None:
        return {**base, "reason": f"could not parse a CDS position from hgvsc={hgvsc!r}"}

    exon = _find_titin_exon(pos, side)
    if exon is None:
        return {**base, "reason": f"CDS position {pos} ({side or 'exonic'}) did not match any titin_exon boundary"}

    is_terminal = exon["te_id"] == _TITIN_LAST_EXON_ID
    psi_dcm = exon["te_psi_dcm"]

    detail = {
        "exon_te_id":       exon["te_id"],
        "exon_domain":      exon["te_domain"],
        "psi_dcm":          psi_dcm,
        "is_terminal_exon": is_terminal,
        "gate_pass":        False,
        "reason":           None,
    }

    if is_terminal:
        return {**detail, "reason": "variant is in TTN's terminal exon -- escapes NMD, PVS1 not supported"}
    if psi_dcm < _TTN_PSI_DCM_THRESHOLD:
        return {**detail, "reason": f"exon PSI={psi_dcm} < {_TTN_PSI_DCM_THRESHOLD} -- not constitutively included, PVS1 not supported"}

    return {**detail, "gate_pass": True}




In [ ]:
# Null consequences from VEP that trigger PVS1 consideration
_NULL_CONSEQUENCES = {
    "transcript_ablation",
    "splice_acceptor_variant",
    "splice_donor_variant",
    "stop_gained",
    "frameshift_variant",
    "start_lost",
}

# G2P variant_class terms that indicate TRUE LOF mechanism.
# Excludes NMD_escaping variants (those produce truncated dominant-negative protein,
# not haploinsufficiency — e.g. MYH7 stop_gained_NMD_escaping).
_LOF_G2P_CLASSES = {
    "transcript_ablation",
    "splice_acceptor_variant",
    "splice_donor_variant",
    "frameshift_variant",
    "frameshift_variant_nmd_triggering",
    "stop_gained",
    "stop_gained_nmd_triggering",
    "start_lost",
}


def _g2p_supports_lof(g2p_hits):
    """
    Return True if ANY G2P entry for this gene lists a true LOF variant class.
    Uses variant_class_with_pathogenicity_evidence (semicolon-separated VEP terms).
    NMD-escaping variants are excluded — they indicate dominant-negative, not LOF.
    """
    if g2p_hits is None or g2p_hits.empty:
        return False
    for _, row in g2p_hits.iterrows():
        classes_raw = str(row.get("variant_class_with_pathogenicity_evidence", ""))
        if classes_raw in ("nan", ""):
            continue
        gene_classes = {c.strip().lower() for c in classes_raw.split(";")}
        if gene_classes & _LOF_G2P_CLASSES:
            return True
    return False


def apply_pvs1_rule(context):
    """
    PVS1 (very strong pathogenic): null variant in a gene where LOF causes disease.

    Condition 1 — variant is a null allele (VEP consequence).
    Condition 2 — Cardiac G2P lists a true LOF variant class for this gene
                  (frameshift, stop_gained, splice donor/acceptor; NOT NMD-escaping
                  which indicates dominant-negative rather than haploinsufficiency).
    Condition 3 (TTN only) — the exon hit is non-terminal and constitutively
                  included (see check_ttn_pvs1_gate above). TTN has no VCEP
                  CSpec, so generic PVS1 would otherwise over-call every null
                  variant regardless of exon usage.

    Examples:
      MYH7 HCM  → G2P lists only missense + stop_gained_NMD_escaping → PVS1 False
      PKP2 ARVC → G2P lists frameshift + stop_gained + splice → PVS1 True (if null variant)
      MYBPC3 HCM → G2P lists frameshift_NMD_triggering + stop_gained_NMD_triggering → PVS1 True
      TTN DCM → null variant, but exon PSI<90 or terminal exon → PVS1 False despite LOF mechanism
    """
    tx = context.get("chosen_transcript") or {}
    consequences = set(tx.get("consequence_terms", []) or [])
    gene_symbol = context.get("gene_symbol")

    is_null   = bool(consequences & _NULL_CONSEQUENCES)
    lof_mech  = _g2p_supports_lof(context.get("g2p_hits"))

    ttn_gate = None
    ttn_supported = True
    if gene_symbol == "TTN" and is_null and lof_mech:
        ttn_gate = check_ttn_pvs1_gate(tx)
        ttn_supported = ttn_gate["gate_pass"]

    return {
        "consequence_terms": sorted(consequences),
        "is_null_variant":   is_null,
        "lof_mechanism":     lof_mech,
        "ttn_gate":          ttn_gate,
        "PVS1":              is_null and lof_mech and ttn_supported,
    }


In [ ]:
_LOF_ONLY_MECHANISMS = {"loss of function"}


def _g2p_mechanism_is_lof_only(g2p_hits):
    """
    True only if every mechanism entry curated for this gene in Cardiac G2P is
    'loss of function' (excludes gain of function / dominant-negative /
    undetermined entries, and genes with no mechanism curated at all) --
    i.e. missense variants are not expected to be pathogenic via this gene's
    known mechanism.
    """
    if g2p_hits is None or g2p_hits.empty or "mechanism" not in g2p_hits.columns:
        return False
    mechanisms = {
        str(m).strip().lower() for m in g2p_hits["mechanism"]
        if str(m).strip().lower() not in ("nan", "na", "")
    }
    if not mechanisms:
        return False
    return mechanisms.issubset(_LOF_ONLY_MECHANISMS)


def apply_bp1_rule(context):
    """
    BP1 (supporting benign): missense variant in a gene where only truncating/
    LOF variants are a known disease mechanism -- i.e. missense is not
    expected to be pathogenic through that mechanism. Mirror image of the
    PVS1 check above, using the same Cardiac_G2P mechanism data.

    Gated on ClinGen CSpec applicability. Confirmed "Not Applicable" for all
    8 ClinGen Cardiomyopathy VCEP genes (MYH7, MYBPC3, TNNI3, TNNT2, TPM1,
    ACTC1, MYL2, MYL3) -- their HCM mechanism is dominant-negative/gain-of-
    function missense, not haploinsufficiency, so BP1 is expected to always
    return False for this panel. It remains usable for other genes in
    Cardiac_G2P.csv that fall outside this VCEP, where no CSpec override
    exists and the generic ACMG BP1 logic (via G2P-curated mechanism)
    applies.
    """
    tx = context.get("chosen_transcript") or {}
    consequences = set(tx.get("consequence_terms") or [])
    gene_symbol = context.get("gene_symbol")
    thresholds = fetch_cspec_thresholds(gene_symbol)
    cspec_applicable = thresholds.get("applicability", {}).get("BP1")

    base = {"BP1": False, "reason": None}

    if cspec_applicable is False:
        return {**base, "reason": "BP1 marked Not Applicable in CSpec for this gene"}

    if "missense_variant" not in consequences:
        return {**base, "reason": "not a missense variant"}

    if not _g2p_mechanism_is_lof_only(context.get("g2p_hits")):
        return {**base, "reason": "gene mechanism is not LOF-only per Cardiac G2P"}

    return {**base, "BP1": True}


In [ ]:
def build_acmg_summary(context):
    pvs1_rules = apply_pvs1_rule(context)
    freq_rules = apply_population_rules(context)
    comp_rules = apply_computational_rules(context)

    evidence = []
    if pvs1_rules["PVS1"]:  evidence.append("PVS1")
    if freq_rules["BA1"]:   evidence.append("BA1")
    if freq_rules["BS1"]:   evidence.append("BS1")
    if freq_rules["PM2"]:   evidence.append("PM2")
    if comp_rules["PP3"]:   evidence.append("PP3")
    if comp_rules["BP4"]:   evidence.append("BP4")

    return evidence


In [ ]:
_CLINVAR_PATHOGENIC = {"pathogenic", "likely_pathogenic"}
_CLINVAR_BENIGN     = {"benign", "likely_benign"}


def apply_clinvar_rules(context):
    """
    PP5: ClinVar classifies variant as Pathogenic or Likely Pathogenic (supporting pathogenic).
    BP6: ClinVar classifies variant as Benign or Likely Benign (supporting benign).

    Uses clin_sig from VEP colocated_variants, saved in annotation as clinvar_significance.
    Only one of PP5/BP6 can fire; conflicting ClinVar reports → neither fires.
    """
    annotation = context.get("annotation") or {}
    sig = annotation.get("clinvar_significance")

    rules = {
        "clinvar_significance": sig,
        "PP5": False,
        "BP6": False,
    }

    if sig in _CLINVAR_PATHOGENIC:
        rules["PP5"] = True
    elif sig in _CLINVAR_BENIGN:
        rules["BP6"] = True

    return rules


In [ ]:
import re
import requests

NCBI_EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

_PM5_CACHE = {}

_AA3_TO_1 = {
    "Ala": "A", "Arg": "R", "Asn": "N", "Asp": "D", "Cys": "C",
    "Gln": "Q", "Glu": "E", "Gly": "G", "His": "H", "Ile": "I",
    "Leu": "L", "Lys": "K", "Met": "M", "Phe": "F", "Pro": "P",
    "Ser": "S", "Thr": "T", "Trp": "W", "Tyr": "Y", "Val": "V",
    "Ter": "*",
}

_CLEARLY_PATHOGENIC = {"pathogenic", "likely pathogenic", "pathogenic/likely pathogenic"}

# Perl equivalent required count_sigs>1 (>=2 corroborating ClinVar submissions
# calling the variant Pathogenic) AND count_confl<1 (zero conflicting Benign/
# Likely Benign submissions) before PS1/PM5 could fire -- a single, unreviewed
# submission was NOT enough (it fell into an inert "reserve" bucket instead).
# Modern ClinVar's germline_classification.description is already an
# aggregate call, so we approximate "corroborated, non-conflicting" via
# review_status instead of raw submitter counts.
_CORROBORATED_REVIEW_STATUSES = {
    "criteria provided, multiple submitters, no conflicts",
    "reviewed by expert panel",
    "practice guideline",
}


def _parse_hgvsp(hgvsp):
    """
    Extract (ref1, position, alt1) in single-letter AA codes from hgvsp.
    e.g. 'NP_000248.2:p.Glu797Lys' -> ('E', 797, 'K')
    """
    if not hgvsp:
        return None, None, None
    m = re.search(r"p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2}|Ter|\*)", hgvsp)
    if not m:
        return None, None, None
    ref1 = _AA3_TO_1.get(m.group(1), m.group(1)[0])
    alt1 = _AA3_TO_1.get(m.group(3), "*" if m.group(3) in ("Ter","*") else m.group(3)[0])
    return ref1, int(m.group(2)), alt1


def _is_clearly_pathogenic(sig_str):
    """True only for unambiguous Pathogenic / LP (excludes Conflicting, VUS, etc.)."""
    return sig_str.lower().strip() in _CLEARLY_PATHOGENIC


def _is_corroborated(review_status):
    """
    True only if ClinVar's review status reflects agreement across multiple
    submitters (or expert-panel/guideline level review) -- excludes
    "no assertion criteria provided" and "criteria provided, single submitter",
    which is exactly the weak, single-source evidence Perl's count_sigs>1
    check was designed to exclude.
    """
    return str(review_status or "").strip().lower() in _CORROBORATED_REVIEW_STATUSES


_STRENGTH_TIER_FULL        = {"pathogenic", "pathogenic/likely pathogenic"}
_STRENGTH_TIER_LIKELY_ONLY = {"likely pathogenic"}


def _clinvar_comparator_strength(hits, tier_full, tier_likely_only):
    """
    Per ClinGen SVI (2023 recommendations building on the original 2015
    framework): PS1/PM5 fire at full strength when the comparator variant's
    own ClinVar classification is (strictly) Pathogenic, but are downgraded
    one strength level when the comparator is only Likely Pathogenic -- a
    weaker anchor still counts, just for less.
      PS1: Pathogenic comparator -> Strong;   Likely Pathogenic only -> Moderate
      PM5: Pathogenic comparator -> Moderate; Likely Pathogenic only -> Supporting
    Returns None if hits is empty.
    """
    sigs = {str(h["significance"]).lower().strip() for h in hits}
    if sigs & _STRENGTH_TIER_FULL:
        return tier_full
    if sigs & _STRENGTH_TIER_LIKELY_ONLY:
        return tier_likely_only
    return None


def _fetch_clinvar_at_position(gene_symbol, ref1aa, position):
    """
    Targeted ClinVar search for variants at a specific amino acid position.
    ClinVar indexes 1-letter AA codes in its text fields — searches
    '{ref1aa}{position}' (e.g. 'E797') across all ClinVar text for this gene.
    Returns list of (ref1, pos, alt1, sig, cdna_change) tuples for clearly
    P/LP entries that are ALSO corroborated (multiple submitters / expert
    panel / practice guideline review status) -- see _is_corroborated.
    cdna_change (e.g. "c.1505G>A") comes from ClinVar's own variation_set,
    letting callers exclude a hit that is actually the query variant's own
    record (same protein change, but NOT an independent nucleotide change --
    see apply_ps1_rule).
    Results cached per (gene, ref1, position).
    """
    cache_key = f"{gene_symbol}:{ref1aa}{position}"
    if cache_key in _PM5_CACHE:
        return _PM5_CACHE[cache_key]

    term = f'"{gene_symbol}"[gene] AND "{ref1aa}{position}"'
    try:
        sr = requests.get(
            f"{NCBI_EUTILS}/esearch.fcgi",
            params={"db": "clinvar", "term": term,
                    "retmax": 100, "retmode": "json"},
            timeout=30,
        )
        sr.raise_for_status()
        ids = sr.json().get("esearchresult", {}).get("idlist", [])
        if not ids:
            _PM5_CACHE[cache_key] = []
            return []

        er = requests.get(
            f"{NCBI_EUTILS}/esummary.fcgi",
            params={"db": "clinvar", "id": ",".join(ids[:100]),
                    "retmode": "json"},
            timeout=30,
        )
        er.raise_for_status()
        result = er.json().get("result", {})

        hits = []
        for uid, doc in result.items():
            if uid == "uids":
                continue
            gc  = doc.get("germline_classification") or {}
            sig = str(gc.get("description", "") or "")
            if not _is_clearly_pathogenic(sig):
                continue
            if not _is_corroborated(gc.get("review_status")):
                continue
            variation_set = doc.get("variation_set") or [{}]
            cdna_change = str((variation_set[0] or {}).get("cdna_change", "") or "")
            pc = str(doc.get("protein_change", "") or "")
            for seg in pc.split(","):
                m = re.match(r"([A-Z])(\d+)([A-Z]|\*)", seg.strip())
                if m and int(m.group(2)) == position:
                    hits.append((m.group(1), int(m.group(2)), m.group(3), sig, cdna_change))

        _PM5_CACHE[cache_key] = hits
        return hits

    except Exception as exc:
        print(f"PM5 ClinVar fetch warning ({gene_symbol} {ref1aa}{position}): {exc}")
        _PM5_CACHE[cache_key] = []
        return []


def apply_pm5_rule(context):
    """
    PM5 (moderate pathogenic): Novel missense at an amino acid residue where a
    different missense change determined to be pathogenic has been seen before.

    Algorithm:
      1. Variant must be a missense.
      2. Parse protein position + AA change from VEP hgvsp (e.g. p.Glu797Lys).
      3. Targeted ClinVar text search for that gene + ref1AA + position (e.g. "E797").
      4. PM5 = True if a clearly P/LP, corroborated entry with a DIFFERENT alt
         AA is found (see _is_corroborated -- single-submitter, unreviewed
         ClinVar entries do not count, matching the Perl count_sigs>1 gate).
    """
    tx = context.get("chosen_transcript") or {}
    consequences = set(tx.get("consequence_terms") or [])
    base = {"PM5": False, "PM5_strength": None, "pm5_hits": [], "position": None, "current_change": None}

    if "missense_variant" not in consequences:
        return {**base, "reason": "not a missense variant"}

    gene_symbol = context.get("gene_symbol")
    hgvsp = tx.get("hgvsp", "") or ""
    ref1, position, alt1 = _parse_hgvsp(hgvsp)

    if position is None:
        return {**base, "reason": "could not parse protein position from hgvsp"}

    current_change = f"{ref1}{position}{alt1}"

    clinvar_hits = _fetch_clinvar_at_position(gene_symbol, ref1, position)

    pm5_hits = []
    seen = set()
    for hit_ref, hit_pos, hit_alt, hit_sig, _hit_cdna in clinvar_hits:
        hit_change = f"{hit_ref}{hit_pos}{hit_alt}"
        if hit_change == current_change or hit_change in seen:
            continue
        seen.add(hit_change)
        pm5_hits.append({"protein_change": hit_change, "significance": hit_sig})

    pm5_strength = _clinvar_comparator_strength(
        pm5_hits, tier_full="moderate", tier_likely_only="supporting"
    )

    return {
        "PM5":            len(pm5_hits) > 0,
        "PM5_strength":   pm5_strength,
        "pm5_hits":       pm5_hits[:5],
        "gene":           gene_symbol,
        "position":       position,
        "current_change": current_change,
        "reason":         None,
    }


In [ ]:
def apply_ps1_rule(context):
    """
    PS1 (strong pathogenic): same amino acid change as a variant already
    established as Pathogenic/Likely Pathogenic in ClinVar (different codon/
    nucleotide change, same resulting protein change).

    Reuses the PM5 ClinVar-lookup machinery (_parse_hgvsp /
    _fetch_clinvar_at_position) - the only difference from PM5 is that PS1
    requires the SAME alt amino acid at this position, whereas PM5 requires a
    DIFFERENT one.

    Perl explicitly excluded the queried variant's own ClinVar record as a
    corroborator (mut_coding<>'$cdna_variant') - PS1 requires an
    INDEPENDENT, different nucleotide change producing the same protein
    change, not the variant re-corroborating itself. Because PS1's match
    criterion is "same protein change", that self-record can't be filtered
    out by protein change alone (unlike PM5, which requires a DIFFERENT
    protein change and so can never self-match); it's excluded here by
    comparing normalized cDNA instead.

    Gated on ClinGen CSpec applicability where a VCEP spec exists for this
    gene. Confirmed "Applicable" for all 8 Cardiomyopathy VCEP genes (generic
    Richards 2015 rule, no gene-specific numeric threshold).

    Known limitation (per CSpec, e.g. GN095/MYBPC3): PS1 should not be used
    when the comparator variant's real mechanism is splicing disruption
    (e.g. NMD) rather than a true missense effect - that distinction isn't
    checked here and would need a dedicated splicing assessment.
    """
    tx = context.get("chosen_transcript") or {}
    consequences = set(tx.get("consequence_terms") or [])
    gene_symbol = context.get("gene_symbol")
    thresholds = fetch_cspec_thresholds(gene_symbol)
    cspec_applicable = thresholds.get("applicability", {}).get("PS1")

    base = {"PS1": False, "PS1_strength": None, "ps1_hits": [], "position": None,
            "current_change": None, "reason": None}

    if cspec_applicable is False:
        return {**base, "reason": "PS1 marked Not Applicable in CSpec for this gene"}

    if "missense_variant" not in consequences:
        return {**base, "reason": "not a missense variant"}

    hgvsp = tx.get("hgvsp", "") or ""
    ref1, position, alt1 = _parse_hgvsp(hgvsp)

    if position is None:
        return {**base, "reason": "could not parse protein position from hgvsp"}

    current_change = f"{ref1}{position}{alt1}"
    query_cdna = _bare_cdna(tx.get("hgvsc"))
    clinvar_hits = _fetch_clinvar_at_position(gene_symbol, ref1, position)

    ps1_hits = []
    seen = set()
    for hit_ref, hit_pos, hit_alt, hit_sig, hit_cdna in clinvar_hits:
        hit_change = f"{hit_ref}{hit_pos}{hit_alt}"
        if hit_change != current_change or hit_change in seen:
            continue
        if query_cdna and _bare_cdna(hit_cdna) == query_cdna:
            continue  # this "corroborator" is the query variant's own ClinVar record
        seen.add(hit_change)
        ps1_hits.append({"protein_change": hit_change, "significance": hit_sig})

    ps1_strength = _clinvar_comparator_strength(
        ps1_hits, tier_full="strong", tier_likely_only="moderate"
    )

    return {
        "PS1":            len(ps1_hits) > 0,
        "PS1_strength":   ps1_strength,
        "ps1_hits":       ps1_hits[:5],
        "position":       position,
        "current_change": current_change,
        "reason":         None,
    }


In [ ]:
_PM1_STRENGTH_RANK = {"moderate": 2, "supporting": 1}


def apply_pm1_rule(context):
    """
    PM1 (moderate, or supporting for weaker hotspots): variant falls within a
    curated mutational hotspot / functional domain for this gene.
    PP2 (supporting): whole-gene rule for genes with a high etiological fraction
    of non-truncating variants, where no specific hotspot sub-region is defined
    (a row in hotspot_regions.csv with rule=='PP2' and no aa range, e.g. TPM1).

    Source: hotspot_regions.csv, the flat-file equivalent of the old ICC
    domain_analysis/domain_data tables, with the Fisher's-p/enrichment-factor
    threshold check already resolved into the rule + strength columns.

    NOTE: lookup is by gene only, not gene+disease. In the current reference
    file, genes that appear under more than one disease (MYH7, TNNT2) only
    have a real (non-NULL) region defined for ONE of those diseases -- the
    other is a NULL row and contributes nothing. If the file is ever extended
    with genuinely conflicting regions for the same gene under different
    diseases, this function will need an explicit disease argument to
    disambiguate.
    """
    tx = context.get("chosen_transcript") or {}
    consequences = set(tx.get("consequence_terms") or [])
    base = {
        "PM1": False, "PM1_strength": None,
        "PP2": False, "PP2_strength": None,
        "matched_region": None, "pmid": None,
        "have_assessed": False, "reason": None,
    }

    if "missense_variant" not in consequences:
        return {**base, "reason": "not a missense variant"}

    gene_symbol = context.get("gene_symbol")
    hgvsp = tx.get("hgvsp", "") or ""
    _, position, _ = _parse_hgvsp(hgvsp)

    if position is None:
        return {**base, "reason": "could not parse protein position from hgvsp"}

    gene_hits = lookup_hotspot_by_gene(gene_symbol, hotspot_clean)
    if gene_hits.empty:
        return {**base, "reason": "gene not in hotspot reference file"}

    real_rows = gene_hits[gene_hits["rule"].isin(["PM1", "PP2"])]
    if real_rows.empty:
        # only NULL rows present -> gene/disease assessed, no hotspot applicable
        return {**base, "have_assessed": True,
                "reason": "no hotspot region defined for this gene"}

    # 1. Position-specific PM1 regions
    pm1_rows = real_rows[real_rows["rule"] == "PM1"].dropna(subset=["aa_start", "aa_end"])
    matches = pm1_rows[
        (pm1_rows["aa_start"] <= position) & (position <= pm1_rows["aa_end"])
    ]

    if not matches.empty:
        # if regions overlap, prefer the stronger call (moderate over supporting)
        matches = matches.assign(
            _rank=matches["strength"].map(_PM1_STRENGTH_RANK).fillna(0)
        ).sort_values("_rank", ascending=False)
        best = matches.iloc[0]
        return {
            **base,
            "PM1":            True,
            "PM1_strength":   best["strength"],
            "matched_region": best["description"],
            "pmid":           best["PMID"],
            "have_assessed":  True,
        }

    # 2. Whole-gene PP2 (no coordinates -> applies gene-wide, not by residue)
    pp2_rows = real_rows[(real_rows["rule"] == "PP2") & (real_rows["aa_start"].isna())]
    if not pp2_rows.empty:
        best = pp2_rows.iloc[0]
        return {
            **base,
            "PP2":            True,
            "PP2_strength":   best["strength"],
            "matched_region": best["description"],
            "pmid":           best["PMID"],
            "have_assessed":  True,
        }

    # Position-specific rows exist for this gene, but this residue isn't in any of them
    return {**base, "have_assessed": True,
            "reason": "residue not within any curated hotspot region"}


In [ ]:
_PM4_BP3_CONSEQUENCES = {"inframe_insertion", "inframe_deletion", "stop_lost"}

ENSEMBL_OVERLAP_API = "https://rest.ensembl.org/overlap/region/human"
_REPEAT_CACHE = {}


def _fetch_repeat_class(chrom, start, end, genome="hg38"):
    """
    Query Ensembl's own overlap/region endpoint (feature=repeat) for any
    repeat feature overlapping this genomic window -- same REST server
    (rest.ensembl.org) annotation.ipynb's VEP calls already use, replacing
    the separate UCSC RepeatMasker API this used to call (one fewer external
    service the pipeline depends on).

    Returns the specific repeat name (e.g. "AluY", "L1ME2", "MIR3") from
    Ensembl's own RepeatMasker-derived annotation, or None. This is more
    specific than UCSC's repClass (e.g. "SINE"/"LINE"), but apply_pm4_bp3_rule
    below only ever checks truthiness (any repeat present -> BP3), never
    branches on which class, so the swap is a clean drop-in -- and the
    specific repeat name is arguably more informative in the printed reason
    than UCSC's broader class would have been.

    genome="hg38" kept as a parameter for interface compatibility with the
    UCSC-era signature, though Ensembl's default assembly is already GRCh38
    (confirmed: assembly_name "GRCh38" in live responses), so it's unused here.
    """
    cache_key = (chrom, start, end, genome)
    if cache_key in _REPEAT_CACHE:
        return _REPEAT_CACHE[cache_key]

    chrom_ensembl = str(chrom)[3:] if str(chrom).startswith("chr") else str(chrom)
    region = f"{chrom_ensembl}:{start}-{end}"
    try:
        resp = requests.get(
            f"{ENSEMBL_OVERLAP_API}/{region}",
            params={"feature": "repeat"},
            headers={"Content-Type": "application/json"},
            timeout=30,
        )
        resp.raise_for_status()
        features = resp.json() or []
        repeat_class = features[0].get("description") if features else None
    except Exception as exc:
        print(f"Ensembl repeat overlap warning ({region}): {exc}")
        repeat_class = None

    _REPEAT_CACHE[cache_key] = repeat_class
    return repeat_class


def _parse_gnomad_variant_id(gnomad_variant_id):
    """'14-23894525-C-T' -> ('14', 23894525, 'C', 'T')"""
    if not gnomad_variant_id:
        return None, None, None, None
    parts = gnomad_variant_id.split("-")
    if len(parts) != 4:
        return None, None, None, None
    chrom, pos, ref, alt = parts
    try:
        pos = int(pos)
    except ValueError:
        return None, None, None, None
    return chrom, pos, ref, alt


def apply_pm4_bp3_rule(context):
    """
    PM4 (moderate pathogenic): protein length change from an in-frame indel
    or stop-loss variant, outside a repetitive/low-complexity region.
    BP3 (supporting benign): the same in-frame indel falls inside a
    repetitive region with no known function.

    Perl equivalent (Classifier_new.pl): only evaluated for Inframe_Indel or
    Stop_Lost vartypes. For Inframe_Indel, looked up the genomic position in
    ICC's repeat_regions table (rr_chr/rr_start/rr_end) -- BP3 if a repeat
    was found, PM4 otherwise (mutually exclusive, one lookup decides both).
    Stop_Lost always got PM4 unconditionally, no repeat check. Here the
    lookup is Ensembl's own overlap/region API via a live query instead of a local table.

    Gated on ClinGen CSpec applicability. PM4 confirmed "Applicable" for all
    8 Cardiomyopathy VCEP genes (generic Richards 2015 rule -- CSpec text
    suggests the strength may need downgrading to Moderate/Supporting
    depending on indel size/location/conservation, which isn't automated
    here, so it's always applied at the WEIGHTS default). BP3 confirmed
    "Not Applicable" for all 8 VCEP genes -- implemented for other
    Cardiac_G2P.csv genes where no CSpec override exists.
    """
    tx = context.get("chosen_transcript") or {}
    consequences = set(tx.get("consequence_terms") or [])
    gene_symbol = context.get("gene_symbol")
    thresholds = fetch_cspec_thresholds(gene_symbol)
    applicability = thresholds.get("applicability", {})

    base = {"PM4": False, "BP3": False, "repeat_class": None, "reason": None}

    eligible = consequences & _PM4_BP3_CONSEQUENCES
    if not eligible:
        return {**base, "reason": "not an in-frame indel or stop-loss variant"}

    if "stop_lost" in eligible:
        if applicability.get("PM4") is False:
            return {**base, "reason": "PM4 marked Not Applicable in CSpec for this gene"}
        return {**base, "PM4": True,
                "reason": "stop-loss variant (no repeat check, per original logic)"}

    # in-frame insertion/deletion -> need genomic coordinates for repeat lookup
    chrom, pos, ref, alt = _parse_gnomad_variant_id(context.get("gnomad_variant_id"))
    if pos is None:
        return {**base, "reason": "no genomic coordinates available for repeat lookup"}

    indel_len = max(len(ref or ""), len(alt or ""))
    repeat_class = _fetch_repeat_class(chrom, pos - 1, pos + indel_len)

    if repeat_class:
        if applicability.get("BP3") is False:
            # BP3 being CSpec Not Applicable for this gene (true for all 8
            # Cardiomyopathy VCEP genes, per their dominant-negative
            # mechanism) doesn't mean an in-frame indel loses its PM4
            # credit -- the repeat-region check only ever existed to choose
            # BETWEEN PM4 and BP3, so when BP3 itself doesn't apply here,
            # fall through to PM4 exactly as if no repeat had been found.
            if applicability.get("PM4") is False:
                return {**base, "repeat_class": repeat_class,
                        "reason": "in a repeat region, but neither PM4 nor BP3 is applicable in CSpec for this gene"}
            return {**base, "PM4": True, "repeat_class": repeat_class}
        return {**base, "BP3": True, "repeat_class": repeat_class}

    if applicability.get("PM4") is False:
        return {**base, "reason": "PM4 marked Not Applicable in CSpec for this gene"}
    return {**base, "PM4": True}


In [ ]:
_BP7_ELIGIBLE_CONSEQUENCES = {"synonymous_variant"}


def apply_bp7_rule(context):
    """
    BP7 (supporting benign): synonymous variant with no predicted splicing
    impact (SpliceAI delta score below threshold).

    Scope note: CSpec's BP7 text also covers intronic variants outside the
    splice consensus sequence (-4/+7) that are additionally "not highly
    conserved" -- that intronic + conservation branch is NOT implemented
    here (it needs intron-distance parsing from HGVSc and a conservation
    score this pipeline doesn't fetch). Only the synonymous-variant branch
    is automated.

    No CSpec-specific numeric SpliceAI threshold was found for the
    Cardiomyopathy VCEP genes (the ruleset text is qualitative: "splicing
    prediction algorithms predict no impact"). Falls back to the field-
    standard SpliceAI low-precision/high-recall cutoff (< 0.10) from
    Jaganathan et al. 2019.

    The SpliceAI score comes from VEP's own SpliceAI plugin (?SpliceAI=1 in
    annotation.ipynb's _vep_call) -- max delta score across acceptor/donor
    gain/loss, read off the chosen transcript.
    """
    tx = context.get("chosen_transcript") or {}
    consequences = set(tx.get("consequence_terms") or [])
    gene_symbol = context.get("gene_symbol")
    thresholds = fetch_cspec_thresholds(gene_symbol)
    cspec_applicable = thresholds.get("applicability", {}).get("BP7")
    bp7_max = thresholds.get("bp7_max", GENERIC_ACMG_THRESHOLDS.get("bp7_max", 0.10))

    base = {"BP7": False, "spliceai_score": None, "bp7_threshold": bp7_max, "reason": None}

    if cspec_applicable is False:
        return {**base, "reason": "BP7 marked Not Applicable in CSpec for this gene"}

    if not (consequences & _BP7_ELIGIBLE_CONSEQUENCES):
        return {**base, "reason": "not a synonymous variant (intronic branch not implemented)"}

    score = context.get("spliceai_score")
    if score is None:
        return {**base, "reason": "SpliceAI score not available"}

    return {**base, "spliceai_score": score, "BP7": score < bp7_max}


In [ ]:
def apply_ps2_pm6_rule(context):
    """
    PS2 (strong pathogenic): de novo variant in a patient with the disease and
    no family history, with biological parentage (maternity AND paternity)
    confirmed by testing.
    PM6 (moderate pathogenic): assumed de novo, but parentage not confirmed.

    NOT computable from any public API or reference file -- requires clinical/
    family-testing information supplied by the user. Collected interactively
    in main1.ipynb into context["manual_evidence"]["de_novo_status"], one of:
      "confirmed"    -> de novo AND parentage confirmed        -> PS2
      "assumed"      -> de novo, parentage NOT confirmed        -> PM6
      "not_de_novo"  -> variant found in a parent
      "unknown"      -> not assessed / parents not tested       -> neither fires

    Per ClinGen CSpec (e.g. GN002/MYH7): both PS2 and PM6 are "Applicable"
    with no numeric threshold -- actual strength depends on phenotype
    specificity and the number of independent de novo occurrences per SVI
    guidance, which needs clinical judgment beyond what this pipeline
    automates. This function applies only the baseline strength (Strong for
    PS2, Moderate for PM6); it does not attempt an upgrade to Very Strong for
    multiple independent occurrences.
    """
    gene_symbol = context.get("gene_symbol")
    thresholds = fetch_cspec_thresholds(gene_symbol)
    applicability = thresholds.get("applicability", {})

    manual = context.get("manual_evidence") or {}
    status = manual.get("de_novo_status", "unknown")

    base = {"PS2": False, "PM6": False, "de_novo_status": status, "reason": None}

    if status in ("unknown", "not_de_novo"):
        return {**base, "reason": f"de novo status: {status}"}

    if status == "confirmed":
        if applicability.get("PS2") is False:
            return {**base, "reason": "PS2 marked Not Applicable in CSpec for this gene"}
        return {**base, "PS2": True}

    if status == "assumed":
        if applicability.get("PM6") is False:
            return {**base, "reason": "PM6 marked Not Applicable in CSpec for this gene"}
        return {**base, "PM6": True}

    return {**base, "reason": f"unrecognised de_novo_status: {status!r}"}


In [ ]:
import csv
import math
import re
from functools import lru_cache

_PS4_CASE_FILES = {
    "dcm": "JUL_DCMgenes.tsv",
    "hcm-fam": "JUL_HCMgenes.tsv",
    "hcm-syn": "JUL_HCMgenes.tsv",
}

# ClinGen worked example for sarcomeric cardiomyopathy genes (cardiodb.org
# PS4 calculator, https://www.cardiodb.org/ps4_calculator/): grade PS4 by the
# lower bound of the 95% CI on the odds ratio, mirroring the strength-tiering
# already applied to PS1/PM5/PM1/PP2 elsewhere in this pipeline.
_PS4_CI_LOWER_THRESHOLDS = [
    (20, "strong"),
    (10, "moderate"),
    (5, "supporting"),
]


@lru_cache(maxsize=None)
def _load_case_control_index(tsv_path):
    """
    Index a case-cohort AC/AN TSV (contig, pos, ref, alt, AC, AN) by
    (chrom, pos, ref, alt) for O(1) variant lookup. Chrom is stored without
    a 'chr' prefix to match the gnomad_variant_id convention used elsewhere
    in this pipeline (built from VEP's seq_region_name).

    Also tracks, per chromosome, the min/max covered position -- so a lookup
    miss can be told apart from "this gene/region isn't in the cohort at
    all" (e.g. MYBPC3/MYL2/MYL3 are entirely absent from the DCM file) vs
    "the region was sequenced but this exact allele wasn't observed in
    cases". Both mean PS4 doesn't fire, but they are not the same evidence
    claim and the reason string should say which one happened.
    """
    index = {}
    span = {}
    with open(tsv_path, newline="") as fh:
        reader = csv.DictReader(fh, delimiter="\t")
        for row in reader:
            chrom = row["contig"][3:] if row["contig"].startswith("chr") else row["contig"]
            pos = int(row["pos"])
            key = (chrom, row["pos"], row["ref"], row["alt"])
            index[key] = (int(row["AC"]), int(row["AN"]))
            lo, hi = span.get(chrom, (pos, pos))
            span[chrom] = (min(lo, pos), max(hi, pos))
    return index, span


def _region_covered(chrom, pos, tsv_path, margin=100_000):
    """
    Best-effort check for whether this genomic neighbourhood is represented
    in the case cohort at all. No gene model is available for this TSV (it's
    a raw contig/pos/ref/alt/AC/AN dump), so this uses a generous position
    window around the nearest covered variants on the same chromosome as a
    proxy for "this gene was part of the panel that produced this file".
    """
    _, span = _load_case_control_index(tsv_path)
    lo_hi = span.get(chrom)
    if lo_hi is None:
        return False
    lo, hi = lo_hi
    return (lo - margin) <= pos <= (hi + margin)


def _lookup_case_ac_an(gnomad_variant_id, tsv_path):
    if not gnomad_variant_id:
        return None
    parts = gnomad_variant_id.split("-")
    if len(parts) != 4:
        return None
    chrom, pos, ref, alt = parts
    index, _ = _load_case_control_index(tsv_path)
    return index.get((chrom, pos, ref, alt))


_ORIGINAL_CASE_STUDIES_PATH = "original_case_control_studies.csv"
_PS4_DISEASE_CATEGORY = {"dcm": "DCM", "hcm-fam": "HCM", "hcm-syn": "HCM"}


def _normalize_cdna_for_matching(cdna):
    """
    'c.2539_2541delAAG' -> 'c.2539_2541del'; 'c.2539_2541del' unchanged.

    VEP reports pure deletions
    using current HGVS recommendations (no deleted bases spelled out --
    "c.2539_2541del"), while the original legacy database (built
    ~2014-2016) used the older convention of including them
    ("c.2539_2541delAAG"). An exact string match between the two silently
    finds nothing even though it's the same variant, as seen for
    c.2539_2541delAAG, c.2791_2793delGAG, c.3658_3660delGAG, all of which
    have real case data in original_case_control_studies.csv that a bare
    string match never found. Duplications have the same legacy-notation
    issue ("dup" + bases vs bare "dup"); insertions are NOT touched here,
    since the inserted bases are the actual novel content, not redundant
    context, and stripping them would be wrong.
    """
    if not cdna:
        return cdna
    return re.sub(r"(del|dup)[ACGTacgt]+$", r"\1", cdna)


@lru_cache(maxsize=None)
def _load_original_case_studies(path=_ORIGINAL_CASE_STUDIES_PATH):
    """
    Index the original case-cohort data from the legacy tool's internal
    database (variant_study/study/gene_study/mutation tables, see
    original_case_control_studies.csv) -- by (gene, bare cDNA) -> list
    of {disease, case_count, sample_count, study_source} rows. Contributing
    labs: LMM, RBH (Royal Brompton Hospital), ORGL, and others, each an
    independently published cohort.

    This is a SEPARATE, ADDITIONAL case source to JUL_HCMgenes.tsv/
    JUL_DCMgenes.tsv, not a replacement for it. For example, MYH7
    c.1207C>T/p.R403W: JUL_HCMgenes.tsv alone gives
    case_ac=1 (case_an=6300), while the original LMM+ORGL+Other cohorts sum
    to case_ac=13 (case_an=13972) for the exact same variant -- data that
    was sitting unused in the original database dump. Both sources are
    combined in apply_ps4_rule below, matching the real PS4 case data
    volume the original CardioClassifier/ClinGen curation had access to.

    Every study's case_count is summed regardless of that study's OWN
    classification label (e.g. a study calling the variant "VUS" still
    counts as a real observation in an affected patient) -- PS4 asks
    whether the variant is enriched in cases, not whether any one
    historical lab's own conclusion agreed with pathogenicity.
    """
    index = {}
    with open(path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            key = (row["gene"].strip().upper(), _normalize_cdna_for_matching(row["cdna"].strip()))
            index.setdefault(key, []).append({
                "disease":      row["disease"],
                "case_count":   int(row["case_count"]),
                "sample_count": int(row["sample_count"]),
                "study_source": row["study_source"],
            })
    return index


def _lookup_original_case_ac_an(gene_symbol, cdna, disease_category):
    """
    Sum case_count / (2 * sample_count) across every independent original
    cohort study matching this (gene, cdna, disease). Returns None if no
    study covers this exact variant for this disease. Both sides of the
    match are normalized via _normalize_cdna_for_matching (see its
    docstring) so legacy del/dup notation with spelled-out bases still
    matches VEP's current bare notation.
    """
    if not (gene_symbol and cdna and disease_category):
        return None
    key = (gene_symbol.upper(), _normalize_cdna_for_matching(cdna))
    hits = _load_original_case_studies().get(key, [])
    hits = [h for h in hits if h["disease"] == disease_category]
    if not hits:
        return None
    return sum(h["case_count"] for h in hits), sum(2 * h["sample_count"] for h in hits)


_GNOMAD_PS4_TRANSIENT_STATUS = {429, 500, 502, 503, 504}
_GNOMAD_COVERAGE_MIN_MEAN = 20.0
# Observed joint (exome+genome) AN across multiple real, successfully
# resolved MYH7 variants in gnomad_r4: 1,612,628 to 1,614,520 -- tight
# enough to use the midpoint as a representative denominator when a
# variant is confirmed well-covered but has zero observations (see below).
_GNOMAD_REPRESENTATIVE_JOINT_AN = 1_614_000


def _fetch_gnomad_coverage(chrom, pos, dataset="gnomad_r4"):
    """
    Mean exome/genome sequencing depth at chrom:pos (GRCh38). Used only to
    decide whether a gnomAD "Variant not found" response for fetch_gnomad_ac_an
    means a true zero allele count (well-covered position, allele genuinely
    never observed) versus a real coverage gap (position unreliably
    sequenced, so absence is uninformative). No retry here -- this is a
    secondary confirmation check, not a primary lookup; if it fails, the
    calling code falls back to the safe "no data" answer rather than
    guessing.
    """
    query = """
    query Coverage($chrom: String!, $pos: Int!, $dataset: DatasetId!) {
      region(chrom: $chrom, start: $pos, stop: $pos, reference_genome: GRCh38) {
        coverage(dataset: $dataset) { exome { mean } genome { mean } }
      }
    }
    """
    try:
        resp = requests.post(
            GNOMAD_API,
            json={"query": query, "variables": {"chrom": chrom, "pos": int(pos), "dataset": dataset}},
            headers={"Content-Type": "application/json"},
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
    except Exception:
        return None
    coverage = ((data.get("data") or {}).get("region") or {}).get("coverage") or {}
    means = []
    for source in ("exome", "genome"):
        entries = coverage.get(source) or []
        if entries and entries[0].get("mean") is not None:
            means.append(entries[0]["mean"])
    return max(means) if means else None


def fetch_gnomad_ac_an(gnomad_variant_id, dataset="gnomad_r4"):
    """
    Raw allele count/number from gnomAD (genome+exome combined 'joint'
    field) -- the control arm of the PS4 case-control comparison. Distinct
    from fetch_gnomad_faf, which only returns a filtered AF, not raw counts.

    Sourced from the shared, cached fetch_gnomad_variant_data (annotation.ipynb)
    -- originally this ran its own separate HTTP request with its own retry
    loop, duplicating fetch_gnomad_faf's request for the same variant.
    Confirmed real impact: running both per variant sustains enough gnomAD
    request volume across a full batch to trip rate limiting partway
    through (see fetch_gnomad_variant_data's docstring). The retry/backoff
    logic now lives there, once, shared by both callers.
    """
    empty = {"control_ac": None, "control_an": None}
    if not gnomad_variant_id:
        return empty

    data = fetch_gnomad_variant_data(gnomad_variant_id, dataset)

    if data.get("errors"):
        # "Variant not found" is a real, legitimate answer, not a failure
        # worth retrying -- but it is NOT the same claim as "no data".
        # Confirmed real impact (57-variant MYH7 set): 5 variants with
        # substantial independent case evidence (case_ac up to 21) were
        # silently skipping PS4 entirely because this was treated as
        # "unknown", when gnomAD coverage at those exact positions is
        # ~30-50x -- i.e. gnomAD genuinely observed ZERO copies of the
        # allele across ~1.6M alleles, a real and informative zero for the
        # case-control odds ratio (Haldane-Anscombe correction already
        # handles zero cells in _odds_ratio_ci below). Only trust this as a
        # true zero when coverage confirms the position was actually
        # well-sequenced; otherwise "not found" could just mean "not
        # reliably called here" and stays "no data".
        parts = gnomad_variant_id.split("-")
        chrom, pos = (parts[0], parts[1]) if len(parts) == 4 else (None, None)
        mean_coverage = _fetch_gnomad_coverage(chrom, pos, dataset) if chrom else None
        if mean_coverage is not None and mean_coverage >= _GNOMAD_COVERAGE_MIN_MEAN:
            return {"control_ac": 0, "control_an": _GNOMAD_REPRESENTATIVE_JOINT_AN}
        return empty

    joint = ((data.get("data") or {}).get("variant") or {}).get("joint") or {}
    ac, an = joint.get("ac"), joint.get("an")
    if ac is None or an is None:
        return empty
    return {"control_ac": int(ac), "control_an": int(an)}


def _odds_ratio_ci(case_ac, case_an, control_ac, control_an, z=1.96):
    """
    Odds ratio and 95% CI via the standard log-OR normal approximation:
      OR = (a/c) / (b/d),  SE(log OR) = sqrt(1/a + 1/b + 1/c + 1/d)
    where a=case alt, c=case ref, b=control alt, d=control ref.
    Haldane-Anscombe correction (+0.5 to all 4 cells) applied whenever any
    cell is zero, to avoid division by zero / log(0).
    """
    a, c = case_ac, case_an - case_ac
    b, d = control_ac, control_an - control_ac

    if min(a, b, c, d) == 0:
        a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5

    odds_ratio = (a / c) / (b / d)
    se_log_or  = math.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    log_or     = math.log(odds_ratio)

    return {
        "odds_ratio": odds_ratio,
        "ci_lower":   math.exp(log_or - z * se_log_or),
        "ci_upper":   math.exp(log_or + z * se_log_or),
    }


def _diseases_with_case_data_for_gene(all_g2p_hits):
    """
    Every disease_context code (e.g. "hcm-fam", "dcm") this gene is actually
    curated for in Cardiac_G2P, restricted to the ones a PS4 case-cohort
    source exists for at all (_PS4_CASE_FILES). Used ONLY when no specific
    diagnosis was selected -- see apply_ps4_rule's docstring.
    """
    if all_g2p_hits is None or getattr(all_g2p_hits, "empty", True):
        return []
    codes = set()
    for _, row in all_g2p_hits.iterrows():
        matches = disease_ref[disease_ref["referral_indication"] == row.get("referral_indication")]
        if len(matches) > 1:
            matches = matches[matches["hcm_subtype"] == row.get("hcm_subtype")]
        for _, m in matches.iterrows():
            code = str(m["dis_name"]).lower()
            if code in _PS4_CASE_FILES:
                codes.add(code)
    return sorted(codes)


def apply_ps4_rule(context):
    """
    PS4 (strong pathogenic): the prevalence of the variant in affected
    individuals is significantly increased compared with the prevalence in
    controls.

    NOT computable without cohort case data -- needs disease-specific
    case-cohort AC/AN, supplied as JUL_DCMgenes.tsv / JUL_HCMgenes.tsv
    (contig, pos, ref, alt, AC, AN; GRCh38 -- verified against MYH7's known
    GRCh38 coordinate range), selected via
    context["manual_evidence"]["disease_context"] ("dcm"/"hcm"), collected
    interactively in main1.ipynb alongside de novo status and zygosity.

    Control arm: gnomAD joint (genome+exome) AC/AN for the same variant.

    Graded by the lower bound of the 95% CI on the odds ratio, following the
    ClinGen worked example for sarcomeric cardiomyopathy genes (cardiodb.org
    PS4 calculator): lower CI >=20 -> Strong, >=10 -> Moderate, >=5 ->
    Supporting. An OR is only taken to indicate increased risk at all if the
    lower CI exceeds 1; below that (or below the Supporting cutoff) PS4 does
    not fire. This is a more granular, currently-recommended alternative to
    the Perl logic's cruder "any pre-computed p-value present" heuristic.
    """
    base = {
        "PS4": False, "PS4_strength": None, "reason": None,
        "case_ac": None, "case_an": None, "control_ac": None, "control_an": None,
        "odds_ratio": None, "ci_lower": None, "ci_upper": None,
    }

    manual = context.get("manual_evidence") or {}
    disease = manual.get("disease_context")

    if disease:
        '''
        A specific diagnosis was selected (the normal case -- mainn.ipynb
        always requires picking one) -- stay scoped to exactly that
        disease, never broadened. Using a DCM patient's case data as
        evidence for an HCM classification (or vice versa) would be a
        real evidence-type mismatch, not a data-completeness improvement,
        even for a pleiotropic gene like MYH7.
        '''
        candidate_diseases = [disease]
    else:
        '''
        No specific diagnosis available at all -- fall back to every
        disease this gene is actually curated for in Cardiac_G2P, rather
        than giving up with "no case cohort file". Whiffin et al. 2018's
        own MYH7 benchmark explicitly used an "All Cardiomyopathy" scope
        for exactly this reason (MYH7 causes a spectrum of cardiomyopathy
        phenotypes) -- this only applies here in the analogous "we don't
        know which single disease to scope to" situation, not whenever
        one happens to be selected.
        '''
        candidate_diseases = _diseases_with_case_data_for_gene(context.get("all_g2p_hits"))
        if not candidate_diseases:
            return {**base, "reason": "no diagnosis selected, and no PS4 case-cohort disease found for this gene"}

    gnomad_variant_id = context.get("gnomad_variant_id")
    tx = context.get("chosen_transcript") or {}
    gene_symbol = context.get("gene_symbol")
    cdna = _bare_cdna(tx.get("hgvsc"))

    total_case_ac, total_case_an = 0, 0
    any_case_data = False
    any_region_covered = False
    seen_tsv_paths = set()

    for d in candidate_diseases:
        tsv_path = _PS4_CASE_FILES.get(d)
        if tsv_path and tsv_path not in seen_tsv_paths:
            seen_tsv_paths.add(tsv_path)
            tsv_case = _lookup_case_ac_an(gnomad_variant_id, tsv_path)
            if tsv_case is not None:
                total_case_ac += tsv_case[0]; total_case_an += tsv_case[1]; any_case_data = True
            parts = (gnomad_variant_id or "").split("-")
            chrom, pos = (parts[0], int(parts[1])) if len(parts) == 4 else (None, None)
            if chrom is not None and _region_covered(chrom, pos, tsv_path):
                any_region_covered = True

        original_case = _lookup_original_case_ac_an(gene_symbol, cdna, _PS4_DISEASE_CATEGORY.get(d))
        if original_case is not None:
            total_case_ac += original_case[0]; total_case_an += original_case[1]; any_case_data = True

    if not any_case_data:
        disease_label = disease.upper() if disease else "/".join(d.upper() for d in candidate_diseases)
        if not any_region_covered:
            return {**base, "reason": (
                f"this gene/region is not represented in the {disease_label} "
                f"case cohort file at all -- absence here is NOT evidence "
                f"against enrichment, it just means no data was collected"
            )}
        return {**base, "reason": f"region covered but this exact variant was not observed in {disease_label} cases"}

    case_ac, case_an = total_case_ac, total_case_an

    control = fetch_gnomad_ac_an(gnomad_variant_id)
    control_ac, control_an = control["control_ac"], control["control_an"]
    if control_ac is None or control_an is None:
        return {**base, "case_ac": case_ac, "case_an": case_an,
                "reason": "gnomAD control AC/AN unavailable for this variant"}

    stats = _odds_ratio_ci(case_ac, case_an, control_ac, control_an)
    result = {
        **base, "case_ac": case_ac, "case_an": case_an,
        "control_ac": control_ac, "control_an": control_an, **stats,
    }

    if stats["ci_lower"] <= 1:
        return {**result, "reason": "lower 95% CI does not exceed 1 -- no significant enrichment"}

    for threshold, strength in _PS4_CI_LOWER_THRESHOLDS:
        if stats["ci_lower"] >= threshold:
            return {**result, "PS4": True, "PS4_strength": strength}

    return {**result, "reason": "lower 95% CI below Supporting threshold (5)"}


In [ ]:
import csv
from functools import lru_cache

_ACMG_CURATIONS_PATH = "acmg_curations.csv"
_CLINGEN_CURATIONS_PATH = "clingen_curations.csv"
_REPORT_COMMENT_FUNCTIONAL_PATH = "report_comment_functional.csv"


@lru_cache(maxsize=None)
def _load_acmg_curations(path=_ACMG_CURATIONS_PATH):
    """
    Index the ClinGen Cardiomyopathy VCEP curation table (acmg_curations --
    extracted from the acmg_curations table in the legacy tool's internal
    database) by (gene, bare cDNA change). Indexes every
    curated rule code present (PS3/BS3/PP1/PM6/PS2/... -- whatever a
    curator has entered), not just PS3/BS3; callers filter by their own
    code(s) after lookup. See apply_ps3_bs3_rule and
    apply_curator_only_rules below for the two consumers.

    This is the production app's real PS3 source: confirmed against
    ACMG_classifier_VCFupload_withPerl_test.php, which queries this exact
    table (matching genomic position + ac_include='Y') and force-fires
    ps3_res when ac_acmg_rule=='PS3' and ac_activate_rule=='Y'. It's a
    later, richer table than the one Classifier_new.pl itself reads (see
    _load_clingen_curations below) -- per-rule rows with real evidence text,
    PMIDs and a graded strength, rather than one flat Y/N column per code.

    Matched on (gene_symbol, cdna_change) rather than genomic position:
    acmg_curations' ac_chr/ac_pos are GRCh37 (this pipeline is GRCh38-only --
    see the minus-strand build bug fixed for PS4/gnomad_variant_id), but
    ac_cdna is a bare "c.XXX" string relative to each gene's canonical/LRG
    RefSeq transcript (verified: acmg_curations' MYBPC3 entries use
    NM_000256.3/ENST00000545968 numbering, the same canonical transcript VEP
    already resolves in this pipeline) -- build-independent, so it's the
    safe join key. Same pattern already used for PS1/PM5 (ClinVar
    protein-change matching) elsewhere in this pipeline.
    """
    index = {}
    with open(path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            if row["ac_include"] != "Y" or row["ac_activate_rule"] != "Y":
                continue
            key = (row["ac_gene"].strip().upper(), row["ac_cdna"].strip())
            index.setdefault(key, []).append({
                "code":     row["ac_acmg_rule"],
                "strength": (row["ac_strength"] or _CANONICAL_STRENGTH.get(row["ac_acmg_rule"], "strong")).strip().lower(),
                "evidence": row["ac_evidence"],
                "source":   row["ac_source"] or "acmg_curations",
                "disease":  row["ac_disease"],
            })
    return index


_CANONICAL_STRENGTH = {
    # Fixed ACMG/AMP 2015 strength tier for codes that were never graded by
    # SVI (unlike PS1/PM5/PM1/PP2/PS4/PP1, which do have published strength
    # tables). Used whenever a source row carries no explicit strength.
    "PS3": "strong", "BS3": "strong",
    "PP1": "supporting",
    "PM3": "moderate", "PP4": "supporting",
    "BS2": "strong", "BS4": "moderate",  # ClinGen 2023 PP1/BS4 guidance caps BS4 below Strong
    "BP2": "supporting", "BP5": "supporting",
}


@lru_cache(maxsize=None)
def _load_clingen_curations(path=_CLINGEN_CURATIONS_PATH):
    """
    Index clingen_curations (extracted from the legacy tool's internal
    database) by (gene, bare cDNA change). This is the table Classifier_new.pl's
    own core "Replace all with ClinGen curations if they exist" block reads
    -- one row per fully-curated variant, one flat Y/N column per ACMG code
    (cc_pvs1 .. cc_bp7), rather than acmg_curations' one-row-per-rule shape.

    Only three of clingen_curations' 28 code columns carry a strength value
    (cc_pvs1_strength, cc_ps4_strength, cc_pp1_strength) -- every other
    fired code (including BS4/BP2 here) falls back to its fixed ACMG/AMP
    2015 tier via _CANONICAL_STRENGTH, since the source table simply has
    nowhere to record a graded strength for them.

    This was the second curation table found in the ICC dump, after
    acmg_curations -- checking it turned up 2 real BS4 and 2 real BP2 rows
    (all MYH7/HCM) that acmg_curations doesn't have, plus 27 more PP1 rows
    and 4 more PS3 rows overlapping/extending acmg_curations' coverage.
    PM3/PP4/BS2/BP5 have zero 'Y' rows here too -- genuinely no curated
    data for those four exists anywhere in this dataset.
    """
    index = {}
    with open(path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            key = (row["cc_gene"].strip().upper(), row["cc_cdna"].strip())
            index.setdefault(key, []).append({
                "code":     row["rule_code"],
                "strength": (row["strength"] or _CANONICAL_STRENGTH.get(row["rule_code"], "strong")).strip().lower(),
                "evidence": f"Curated {row['cc_class']} classification ({row['cc_disease']})",
                "source":   row["source"],
                "disease":  row["cc_disease"],
            })
    return index


@lru_cache(maxsize=None)
def _load_report_comment_functional(path=_REPORT_COMMENT_FUNCTIONAL_PATH):
    """
    Third PS3/BS3 source: report_comment.rc_function ('Y'=damaging,
    'B'=benign, joined through report/mutation/gene) -- Classifier_new.pl's
    original functional-evidence table, distinct from clingen_curations.
    Sparser still (7 usable rows total) and mostly outside the VCEP panel
    (KCNQ1, DSG2, ANKRD1, JAG1, SCN5A), so it's the last source consulted.
    """
    index = {}
    with open(path, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            code = {"Y": "PS3", "B": "BS3"}.get(row["rc_function"])
            if code is None:
                continue
            key = (row["gene_symbol"].strip().upper(), row["mut_coding"].strip())
            pmid = f"PMID:{row['ref_pubmed']}" if row["ref_pubmed"] else None
            index.setdefault(key, []).append({
                "code":     code,
                "strength": "strong",
                "evidence": row["rc_comment"],
                "source":   pmid or "report_comment",
                "disease":  row["rep_disease"],
            })
    return index


def _bare_cdna(hgvsc):
    """'ENST00000545968.6:c.3811C>T' -> 'c.3811C>T'"""
    if not hgvsc:
        return None
    return hgvsc.split(":")[-1].strip()


def _gather_curation_hits(gene_symbol, cdna):
    """
    Merge curated hits for (gene, cDNA change) across every curation source
    -- acmg_curations, clingen_curations, report_comment -- rather than
    stopping at the first source with any match. Merging (not falling back
    on an "or") matters: a variant curated in one source for one code (e.g.
    PS3 in acmg_curations) must not hide an independent hit in a different
    source for a different code (e.g. BS4 in clingen_curations) on the same
    (gene, cDNA) key. An earlier version of this lookup used an "or" chain
    and had exactly that latent bug, even though no variant in the current
    dataset happened to trigger it.
    """
    key = (gene_symbol.strip().upper(), cdna)
    hits = []
    hits += _load_acmg_curations().get(key, [])
    hits += _load_clingen_curations().get(key, [])
    hits += _load_report_comment_functional().get(key, [])
    return hits


def apply_ps3_bs3_rule(context):
    """
    PS3 (strong pathogenic): well-established in vitro/in vivo functional
    studies show a damaging effect on protein function/gene product.
    BS3 (strong benign): well-established functional studies show no
    damaging effect.

    Deliberately NOT gated on the patient's selected disease context
    (context["manual_evidence"]["disease_context"], used by PS4): a
    functional assay result (e.g. "causes exon skipping and a truncated
    protein") is evidence about the variant's molecular consequence, not
    about disease prevalence in a specific cohort -- ACMG/AMP scopes PS3 to
    the assay's validity, not the requesting patient's phenotype. The
    curation's own disease annotation is still surfaced in 'matched_disease'
    for transparency.

    PS3 and BS3 are mutually exclusive per the ACMG/AMP framework -- if a
    (gene, cDNA change) somehow has both a damaging and a non-damaging
    curated study (conflicting evidence, possibly from two different
    sources), neither fires; the conflict is reported in 'reason' instead
    of silently picking one side.
    """
    base = {
        "PS3": False, "PS3_strength": None,
        "BS3": False, "BS3_strength": None,
        "matched_disease": None, "evidence": [], "sources": [],
        "reason": None,
    }

    gene_symbol = context.get("gene_symbol")
    tx = context.get("chosen_transcript") or {}
    cdna = _bare_cdna(tx.get("hgvsc"))

    if not gene_symbol or not cdna:
        return {**base, "reason": "no gene/cDNA change available to match against curations"}

    hits = _gather_curation_hits(gene_symbol, cdna)

    ps3_hits = [h for h in hits if h["code"] == "PS3"]
    bs3_hits = [h for h in hits if h["code"] == "BS3"]

    if not ps3_hits and not bs3_hits:
        return {**base, "reason": "no functional-study curation found for this variant"}

    if ps3_hits and bs3_hits:
        return {**base, "reason": (
            "conflicting functional-study curations found (both damaging "
            "and non-damaging) -- neither PS3 nor BS3 applied"
        )}

    fired   = ps3_hits or bs3_hits
    code    = fired[0]["code"]
    strength = fired[0]["strength"]

    result = {
        **base,
        "matched_disease": ", ".join(sorted({h["disease"] for h in fired if h["disease"]})),
        "evidence":         [h["evidence"] for h in fired],
        "sources":          [h["source"] for h in fired],
    }
    if code == "PS3":
        result["PS3"], result["PS3_strength"] = True, strength
    else:
        result["BS3"], result["BS3_strength"] = True, strength
    return result


In [ ]:
_CURATOR_ONLY_CODES = ("PM3", "PP1", "PP4", "BS2", "BS4", "BP2", "BP5")

'''
ClinGen Cardiomyopathy VCEP (cspec.genome.network/cspec/ui/svi/doc/GN098):
- PP1 Strong >=7 informative meioses, Moderate >=5, Supporting >=3, <3 does
not meet PP1 at all. These are the exact thresholds the original PHP
curator form documented too (ACMG_add_evidence_ext.php: "STRONG if random
chance <1% (>=7 meioses/segregations)", "MODERATE ... <5% (>=5)",
"SUPPORTING ... <25% (>=3)") -- the current VCEP spec and the decade-old
Perl tool agree on this one. BS4 (non-segregation), per the December 2023
ClinGen SVI PP1/BS4 guidance (PMID 38141607), requires multiple (>=2)
non-segregations to apply at all, and is capped at Moderate strength -
never Strong, since distinguishing true non-segregation from phenocopies
or a second causal variant is inherently less certain than segregation.
'''

_PP1_STRENGTH_BY_MEIOSES = ((7, "strong"), (5, "moderate"), (3, "supporting"))
_BS4_MIN_NONSEGREGATIONS = 2


def _pp1_strength_from_meioses(n):
    """Grade PP1 by informative-meioses count. Returns None if n < 3 or n is None."""
    if n is None:
        return None
    for threshold, strength in _PP1_STRENGTH_BY_MEIOSES:
        if n >= threshold:
            return strength
    return None


def _manual_curator_hits(context):
    """
    Translate the interactively-collected phasing/segregation/phenotype/BS2/
    BP5 answers (main1.ipynb, asked right after de novo status/zygosity)
    into curator-only evidence hits, in the same {code, strength, evidence,
    source} shape _gather_curation_hits returns, so they merge identically
    with file-sourced hits in apply_curator_only_rules below.

    Routing uses this gene's inheritance/penetrance facts already curated in
    Cardiac_G2P (via _g2p_is_biallelic / _g2p_penetrance_note) rather than
    asking the user to know which ACMG code their answer maps to:

      Phasing (is there a second (likely) pathogenic variant in this gene,
      and is it cis or trans with this one?):
        - trans + recessive (biallelic) gene            -> PM3
        - trans + dominant gene, full penetrance expected -> BP2
        - trans + dominant gene, NOT fully penetrant      -> neither fires
          (BP2's dominant clause explicitly requires full penetrance)
        - cis, any gene                                   -> BP2

      Segregation (assessed across the family; does it segregate?):
        - segregates, >=3 informative meioses             -> PP1 (Strong >=7,
          Moderate >=5, Supporting >=3 -- see _pp1_strength_from_meioses);
          if the meioses count isn't given, falls back to "supporting"
        - does not segregate, >=2 non-segregations (or count
          not given)                                        -> BS4, capped
          at "moderate" (never "strong", per ClinGen 2023 SVI guidance)
        - does not segregate, exactly 1 non-segregation     -> no hit (BS4
          requires multiple non-segregations to stand alone)

      PP4 (phenotype highly specific) and BP5 (alternate molecular basis
      found) are asked and applied directly -- no gene-level routing needed.

      BS2 (observed in a phenotype-negative adult, full penetrance
      expected) is only asked in main1.ipynb when _g2p_penetrance_note is
      None; if the user answers unknown/no or the question was skipped,
      simply no BS2 hit is produced here.
    """
    manual = context.get("manual_evidence") or {}
    g2p_hits = context.get("g2p_hits")
    biallelic = _g2p_is_biallelic(g2p_hits)
    penetrance_note = _g2p_penetrance_note(g2p_hits)

    hits = []

    if manual.get("phasing_second_variant") == "yes":
        phase = manual.get("phasing_relationship")
        if phase == "trans" and biallelic:
            hits.append({
                "code": "PM3", "strength": _CANONICAL_STRENGTH["PM3"],
                "evidence": "User-reported: second (likely) pathogenic variant in trans, recessive gene",
                "source": "user-entered", "disease": None,
            })
        elif phase == "trans" and not biallelic and penetrance_note is None:
            hits.append({
                "code": "BP2", "strength": _CANONICAL_STRENGTH["BP2"],
                "evidence": "User-reported: second (likely) pathogenic variant in trans, fully penetrant dominant gene",
                "source": "user-entered", "disease": None,
            })
        elif phase == "cis":
            hits.append({
                "code": "BP2", "strength": _CANONICAL_STRENGTH["BP2"],
                "evidence": "User-reported: second (likely) pathogenic variant in cis",
                "source": "user-entered", "disease": None,
            })

    if manual.get("segregation_assessed") == "yes":
        seg_result = manual.get("segregation_result")
        meioses = manual.get("segregation_meioses")
        if seg_result == "segregates":
            graded = _pp1_strength_from_meioses(meioses)
            if meioses is not None and graded is None:
                pass  # fewer than 3 informative meioses -- does not meet PP1
            else:
                detail = f" ({meioses} informative meioses)" if meioses is not None else ""
                hits.append({
                    "code": "PP1", "strength": graded or _CANONICAL_STRENGTH["PP1"],
                    "evidence": f"User-reported: segregates with disease across family{detail}",
                    "source": "user-entered", "disease": None,
                })
        elif seg_result == "does_not_segregate":
            if meioses is not None and meioses < _BS4_MIN_NONSEGREGATIONS:
                pass  # a single non-segregation does not meet BS4 alone (ClinGen 2023)
            else:
                detail = f" ({meioses} non-segregations)" if meioses is not None else ""
                hits.append({
                    "code": "BS4", "strength": _CANONICAL_STRENGTH["BS4"],
                    "evidence": f"User-reported: lack of segregation with disease across family{detail}",
                    "source": "user-entered", "disease": None,
                })

    if manual.get("phenotype_specific") == "yes":
        hits.append({
            "code": "PP4", "strength": _CANONICAL_STRENGTH["PP4"],
            "evidence": "User-reported: phenotype/family history highly specific for this gene's disease",
            "source": "user-entered", "disease": None,
        })

    if penetrance_note is None and manual.get("healthy_adult_observed") == "yes":
        hits.append({
            "code": "BS2", "strength": _CANONICAL_STRENGTH["BS2"],
            "evidence": "User-reported: observed in a phenotype-negative adult past expected age of onset",
            "source": "user-entered", "disease": None,
        })

    if manual.get("alternate_molecular_basis") == "yes":
        hits.append({
            "code": "BP5", "strength": _CANONICAL_STRENGTH["BP5"],
            "evidence": "User-reported: alternate molecular basis identified for this patient's phenotype",
            "source": "user-entered", "disease": None,
        })

    return hits


def apply_curator_only_rules(context):
    """
    PM3, PP1, PP4, BS2, BS4, BP2, BP5.

    None of these are computed by Classifier_new.pl either. Grepping the
    Perl source for where each variable (pm3, pp1, pp4, bs2, bs4, bp2,
    bp5) gets assigned turns up exactly one place for each: initialised to
    0 near the top of the script, then only ever set to 1 inside the
    curator-override block that reads clingen_curations -- there is no
    computed-from-data path for any of them anywhere else in the file. So
    in the original app these are curator-only criteria too, not something
    the pipeline was ever meant to derive automatically -- consistent with
    why each resists automation:

      PM3 - detected in trans with a (different) pathogenic variant, for a
            recessive disorder -- needs parental phasing data
      PP1 - cosegregates with disease across a pedigree -- needs a family
            study and a LOD score, not sequence-level data
      PP4 - patient's phenotype is highly specific for the gene's disease --
            needs structured clinical phenotyping, not genomic data
      BS2 - observed in a healthy adult for a fully-penetrant early-onset
            disease -- distinct from population AF (already BA1/BS1/PM2);
            needs a curated "seen in an unaffected individual" case, not a
            frequency threshold
      BS4 - lack of segregation with disease in affected family members
      BP2 - observed in trans with a pathogenic variant for a fully
            penetrant dominant gene, or in cis with a pathogenic variant --
            needs phasing data, same blocker as PM3
      BP5 - variant found in a case with an alternate molecular basis for
            disease -- needs clinical case review, not genomic data

    Sourced from the same merged (acmg_curations + clingen_curations +
    report_comment) lookup apply_ps3_bs3_rule uses -- see
    _gather_curation_hits. As of the last data pull: PP1 has 64 rows in
    acmg_curations plus 27 more in clingen_curations; BS4 and BP2 each have
    2 rows, only in clingen_curations (acmg_curations has none for either --
    missed on the first pass through this dataset, since only acmg_curations
    had been mined at that point). PM3, PP4, BS2 and BP5 have zero rows in
    every source checked so far -- they correctly report "no curation
    found" and will fire the moment a curator (or a future data pull) adds
    one, exactly mirroring Classifier_new.pl's un-overridden default of 0.

    PM7/PP6 ("paralogue mapping", Classifier_new.pl lines ~1417-1465) are
    deliberately NOT implemented here: they are a custom, non-ACMG extension
    restricted to a handful of channelopathy genes (KCNQ1, KCNH2, SCN5A,
    ANK2, KCNE1/2, KCNJ2, CACNA1C, RYR1/2) outside this pipeline's
    cardiomyopathy VCEP panel, and PM7/PP6 are not part of the standard
    ACMG/AMP 2015 criteria set at all.
    """
    gene_symbol = context.get("gene_symbol")
    tx = context.get("chosen_transcript") or {}
    cdna = _bare_cdna(tx.get("hgvsc"))

    result = {code: False for code in _CURATOR_ONLY_CODES}
    result.update({f"{code}_strength": None for code in _CURATOR_ONLY_CODES})
    result["evidence"] = {}
    result["sources"] = {}
    result["reason"] = None

    matched = []
    if gene_symbol and cdna:
        matched += [h for h in _gather_curation_hits(gene_symbol, cdna) if h["code"] in _CURATOR_ONLY_CODES]
    matched += _manual_curator_hits(context)

    if not matched:
        result["reason"] = "no curator-only evidence found for this variant (file or user-entered)"
        return result

    for h in matched:
        code = h["code"]
        result[code] = True
        result[f"{code}_strength"] = h["strength"]
        result["evidence"].setdefault(code, []).append(h["evidence"])
        result["sources"].setdefault(code, []).append(h["source"])

    return result


In [ ]:
_STRENGTH_MAGNITUDE = {
    "stand-alone": 8, "very strong": 8, "strong": 4,
    "moderate": 2, "supporting": 1,
}


def calculate_tavtigian_score(evidence_codes):
    """
    Bayesian posterior probability of pathogenicity (Tavtigian et al. 2018 AJHG).

    Evidence weights (exponents):
      PVS=8, PS=4, PM=2, PP=1, BP=-1, BS=-4, BA=-8
    Combined odds = O_PP^(sum of weights), O_PP = 2.0801
    Posterior = (odds * prior) / (odds * prior + (1 - prior)), prior = 0.10

    Thresholds: P >= 0.99 Pathogenic, >= 0.90 Likely Pathogenic,
                P <= 0.001 Benign, <= 0.10 Likely Benign, else VUS
                (VUS further split into vus_subtier: VUS-low/-mid/-high,
                by combined_score 0-1 / 2-3 / 4-5 -- see Tavtigian et al.
                2020 Table 3)

    evidence_codes entries are normally plain strings (e.g. "PM2"), which take
    the default weight for that code below. Some rules (currently PM1/PP2 from
    the hotspot lookup) can fire at a non-default strength -- pass those as
    (code, strength) tuples, e.g. ("PM1", "supporting"), to override the
    weight using the strength magnitude instead of the code's default.
    """
    WEIGHTS = {
        "PVS1": 8,
        "PS1": 4, "PS2": 4, "PS3": 4, "PS4": 4,
        "PM1": 2, "PM2": 2, "PM3": 2, "PM4": 2, "PM5": 2, "PM6": 2,
        "PP1": 1, "PP2": 1, "PP3": 1, "PP4": 1, "PP5": 1,
        "BA1": -8,
        "BS1": -4, "BS2": -4, "BS3": -4, "BS4": -4,
        "BP1": -1, "BP2": -1, "BP3": -1, "BP4": -1,
        "BP5": -1, "BP6": -1, "BP7": -1,
    }

    # 2.0801, not 2.08: Tavtigian et al. 2020 (Human Mutation) states this
    # constant precisely as 2.0801, derived so that integer point totals
    # land exactly on the continuous posterior thresholds (0.001/0.10/
    # 0.90/0.99) this pipeline uses for its final classification. The
    # truncated "2.08" is close enough that it rarely matters -- except at
    # score=6 exactly, where it gives posterior=0.899978 (just under the
    # 0.90 Likely-Pathogenic threshold) instead of the correct 0.900004,
    # silently misclassifying every variant landing at exactly 6 raw
    # points as VUS instead of Likely Pathogenic.
    O_PP    = 2.0801
    P_PRIOR = 0.10

    def _weight_for(item):
        code, strength = item if isinstance(item, tuple) else (item, None)
        if strength:
            magnitude = _STRENGTH_MAGNITUDE.get(str(strength).lower())
            if magnitude is not None:
                sign = 1 if code.upper().startswith("P") else -1
                return sign * magnitude
        return WEIGHTS.get(code, 0)

    combined_score = sum(_weight_for(item) for item in evidence_codes)
    combined_odds  = O_PP ** combined_score
    posterior      = (combined_odds * P_PRIOR) / (combined_odds * P_PRIOR + (1 - P_PRIOR))

    if posterior >= 0.99:
        classification = "Pathogenic"
    elif posterior >= 0.90:
        classification = "Likely Pathogenic"
    elif posterior <= 0.001:
        classification = "Benign"
    elif posterior <= 0.10:
        classification = "Likely Benign"
    else:
        classification = "VUS"

    # Tavtigian et al. 2020 (Human Mutation, the point-scale companion to
    # the 2018 Bayesian paper) scores "Uncertain" as points 0-5 -- six
    # points wide, vs. 1-4 points for every other tier. A VUS at 5 points is
    # one Supporting-level code away from Likely Pathogenic (6 points); a
    # VUS at 1 point is nowhere close. Subdividing the band (not named by
    # Tavtigian et al., but a direct reading of Table 3's point ranges)
    # surfaces that difference instead of collapsing it into one label.
    vus_subtier = None
    if classification == "VUS":
        if combined_score <= 1:
            vus_subtier = "VUS-low"
        elif combined_score <= 3:
            vus_subtier = "VUS-mid"
        else:
            vus_subtier = "VUS-high"

    return {
        "evidence_codes":        [item[0] if isinstance(item, tuple) else item
                                   for item in evidence_codes],
        "combined_score":        combined_score,
        "posterior_probability": round(posterior, 4),
        "classification":        classification,
        "vus_subtier":           vus_subtier,
    }


def classify_variant(context):
    """Run all evidence rules and return full classification result."""
    pvs1_rules   = apply_pvs1_rule(context)
    freq_rules   = apply_population_rules(context)
    comp_rules   = apply_computational_rules(context)
    clin_rules   = apply_clinvar_rules(context)
    pm5_rules    = apply_pm5_rule(context)
    pm1_rules    = apply_pm1_rule(context)
    ps1_rules    = apply_ps1_rule(context)
    bp1_rules    = apply_bp1_rule(context)
    pm4_rules    = apply_pm4_bp3_rule(context)
    bp7_rules    = apply_bp7_rule(context)
    ps2pm6_rules = apply_ps2_pm6_rule(context)
    ps4_rules    = apply_ps4_rule(context)
    ps3_rules    = apply_ps3_bs3_rule(context)
    curator_rules = apply_curator_only_rules(context)

    evidence = []
    if pvs1_rules["PVS1"]:   evidence.append("PVS1")
    if freq_rules["BA1"]:    evidence.append("BA1")
    if freq_rules["BS1"]:    evidence.append("BS1")
    if freq_rules["PM2"]:    evidence.append(("PM2", freq_rules["pm2_strength"]))
    if ps2pm6_rules["PS2"]:  evidence.append("PS2")
    if ps4_rules["PS4"]:     evidence.append(("PS4", ps4_rules["PS4_strength"]))
    if ps3_rules["PS3"]:     evidence.append(("PS3", ps3_rules["PS3_strength"]))
    if ps3_rules["BS3"]:     evidence.append(("BS3", ps3_rules["BS3_strength"]))
    for _code in _CURATOR_ONLY_CODES:
        if curator_rules[_code]:
            evidence.append((_code, curator_rules[f"{_code}_strength"]))
    if ps1_rules["PS1"]:     evidence.append(("PS1", ps1_rules["PS1_strength"]))
    if pm5_rules["PM5"]:     evidence.append(("PM5", pm5_rules["PM5_strength"]))
    if pm1_rules["PM1"]:     evidence.append(("PM1", pm1_rules["PM1_strength"]))
    if pm1_rules["PP2"]:     evidence.append(("PP2", pm1_rules["PP2_strength"]))
    if pm4_rules["PM4"]:     evidence.append("PM4")
    if pm4_rules["BP3"]:     evidence.append("BP3")
    if ps2pm6_rules["PM6"]:  evidence.append("PM6")
    if bp1_rules["BP1"]:     evidence.append("BP1")
    if comp_rules["PP3"]:    evidence.append("PP3")
    if comp_rules["BP4"]:    evidence.append("BP4")
    if bp7_rules["BP7"]:     evidence.append("BP7")
    # PP5/BP6 deliberately excluded from the score: 2018 ClinGen SVI
    # (Biesecker & Harrison) recommends discontinuing these "reputable
    # source" criteria as non-independent evidence. clin_rules is still
    # computed above and included in clin_detail for reference below.

    result = calculate_tavtigian_score(evidence)
    result["pvs1_detail"]   = pvs1_rules
    result["freq_detail"]   = freq_rules
    result["comp_detail"]   = comp_rules
    result["clin_detail"]   = clin_rules
    result["pm5_detail"]    = pm5_rules
    result["pm1_detail"]    = pm1_rules
    result["ps1_detail"]    = ps1_rules
    result["bp1_detail"]    = bp1_rules
    result["pm4_detail"]    = pm4_rules
    result["bp7_detail"]    = bp7_rules
    result["ps2pm6_detail"] = ps2pm6_rules
    result["ps4_detail"]    = ps4_rules
    result["ps3_detail"]    = ps3_rules
    result["curator_detail"] = curator_rules
    return result
